## 19 MAY 2026

## EDA PROJECT ON ORGANIC BAZAAR

## Business Problem:
#### Srikar has a 100 yards space, he wants to grow organic plants in that space. 
#### Confused to sow which seed in which season in a comfortable budget he seeked for guidance.  

## Problem Statement:
#### Scrape the seed product data Analyze data to identify 
#### best selling seed categories in varying prices and seasons.

#### SOURCE WEBSITE

## https://organicbazar.net/

#### Final_DataFrame_Raw is COMBINED CSV FILE 

In [5]:
import pandas as pd
import numpy as np
import re
import requests
from bs4 import BeautifulSoup
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# Common functions:

In [6]:
def functionto_drop_duplicate_from_dataframe(df, name_column="Name_of_Seed"):
    """
    Cleans scraped dataframe safely.

    Steps:
    1. Remove duplicate rows
    2. Drop empty column names if present
    3. Reset index
    4. Drop rows where name column is NaN

    Parameters:
    ----------
    df : pandas DataFrame
        Input dataframe

    name_column : str
        Column used to identify valid records

    Returns:
    -------
    Cleaned DataFrame
    """
    
    # Create copy to avoid modifying original
    df = df.copy()

    # ---------------- REMOVE DUPLICATES ----------------
    df.drop_duplicates(inplace=True)

    # ---------------- DROP EMPTY COLUMN NAME ----------------
    if '' in df.columns:
        df.drop(columns=[''], inplace=True)

    # ---------------- RESET INDEX ----------------
    df.reset_index(drop=True, inplace=True)

    # ---------------- DROP NULL NAME ROWS ----------------
    if name_column in df.columns:
        df.dropna(subset=[name_column], inplace=True)

    # ---------------- FINAL RESET INDEX ----------------
    df.reset_index(drop=True, inplace=True)

    return df

In [7]:
def functionto_missing_value_percentage(df, columns):
    """
    Returns percentage of missing values
    for specified columns.

    Parameters:
    ----------
    df : pandas DataFrame

    columns : list
        List of column names

    Returns:
    -------
    pandas DataFrame
    """

    result = []

    total_rows = len(df)

    for col in columns:

        # Check column exists
        if col in df.columns:

            missing_count = df[col].isna().sum()

            missing_percent = round(
                (missing_count / total_rows) * 100,
                2
            )

            result.append({
                "Column_Name": col,
                "Missing_Count": missing_count,
                "Missing_Percentage": missing_percent
            })

        else:

            result.append({
                "Column_Name": col,
                "Missing_Count": np.nan,
                "Missing_Percentage": np.nan
            })

    return pd.DataFrame(result)

In [8]:
def functionto_drop_column_with_missing_values(df,missing_cols):
    df =df.drop(columns=missing_cols,inplace =True)
    return df

In [9]:
def functionto_drop_high_missing_columns(df, missing_df, threshold=95):
    """
    Drops columns whose missing percentage
    is greater than threshold.

    Parameters:
    ----------
    df : pandas DataFrame

    missing_df : pandas DataFrame
        Output dataframe from missing_value_percentage()

    threshold : int or float
        Missing percentage threshold

    Returns:
    -------
    Cleaned DataFrame
    """

    # Find columns exceeding threshold
    missing_cols = missing_df[
        missing_df['Missing_Percentage'] > threshold
    ]['Column_Name'].tolist()

    # Drop columns safely
    cleaned_df = df.drop(
        columns=missing_cols,
        errors='ignore'
    )

    return cleaned_df

In [10]:
def fill_categorical_nulls_with_mode(df):
    """
    Fills missing values in categorical columns
    using mode.

    Parameters:
    ----------
    df : pandas DataFrame

    Returns:
    -------
    Cleaned DataFrame
    """

    # create copy
    df = df.copy()

    # -----------------------------------------
    # identify categorical columns
    # -----------------------------------------
    categorical_columns = df.select_dtypes(
        include='object'
    ).columns

    # -----------------------------------------
    # fill nulls with mode
    # -----------------------------------------
    for col in categorical_columns:

        mode_value = df[col].mode()

        # check mode exists
        if not mode_value.empty:

            df[col].fillna(
                mode_value[0],
                inplace=True
            )

    return df

In [11]:
def fill_numeric_nulls_with_mean_median(df):
    """
    Fills missing values in numeric columns.

    Logic:
    -------
    If outliers exist:
        fill NaN with median

    Else:
        fill NaN with mean

    Outlier detection:
        IQR Method

    Parameters:
    ----------
    df : pandas DataFrame

    Returns:
    -------
    Cleaned DataFrame
    """

    # create copy
    df = df.copy()

    # -----------------------------------------
    # numeric columns
    # -----------------------------------------
    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    # -----------------------------------------
    # process each numeric column
    # -----------------------------------------
    for col in numeric_columns:

        # skip fully null columns
        if df[col].isna().all():
            continue

        # -----------------------------------------
        # IQR OUTLIER DETECTION
        # -----------------------------------------

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR

        upper_bound = Q3 + 1.5 * IQR

        # identify outliers
        outliers = df[
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        ]

        # -----------------------------------------
        # fill NaN
        # -----------------------------------------

        # if outliers exist -> median
        if len(outliers) > 0:

            fill_value = df[col].median()

        # otherwise -> mean
        else:

            fill_value = df[col].mean()

        # fill missing values
        df[col].fillna(
            fill_value,
            inplace=True
        )

    return df

In [12]:
def treat_outliers_iqr(df, method='cap'):
    """
    Detects and treats outliers in numeric columns
    using IQR method.

    Parameters:
    ----------
    df : pandas DataFrame

    method : str
        'cap'    -> cap outliers
        'remove' -> remove outlier rows

    Returns:
    -------
    Cleaned DataFrame
    """

    # create copy
    df = df.copy()

    # -----------------------------------------
    # numeric columns
    # -----------------------------------------
    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    # -----------------------------------------
    # process each numeric column
    # -----------------------------------------
    for col in numeric_columns:

        # skip fully null columns
        if df[col].isna().all():
            continue

        # -----------------------------------------
        # calculate IQR
        # -----------------------------------------
        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR

        upper_bound = Q3 + 1.5 * IQR

        # -----------------------------------------
        # CAP METHOD
        # -----------------------------------------
        if method == 'cap':

            df[col] = np.where(
                df[col] < lower_bound,
                lower_bound,
                df[col]
            )

            df[col] = np.where(
                df[col] > upper_bound,
                upper_bound,
                df[col]
            )

        # -----------------------------------------
        # REMOVE METHOD
        # -----------------------------------------
        elif method == 'remove':

            df = df[
                (df[col] >= lower_bound) &
                (df[col] <= upper_bound)
            ]

    # reset index
    df.reset_index(drop=True, inplace=True)

    return df

In [13]:
def clean_scraped_dataframe(df):

    # create copy
    df = df.copy()

    # -----------------------------------------
    # remove duplicate rows
    # -----------------------------------------
    df.drop_duplicates(inplace=True)

    # -----------------------------------------
    # drop empty column names
    # -----------------------------------------
    if '' in df.columns:
        df.drop(columns=[''], inplace=True)

    # -----------------------------------------
    # check percentage of null values in each column and drop if percentage is > 95
    # -----------------------------------------
    missing_percent_df = functionto_missing_value_percentage(df, df.columns)
    df = functionto_drop_high_missing_columns(df, missing_percent_df)


    
    # -----------------------------------------
    # remove rows where Name_of_Seed is null
    # -----------------------------------------
    if 'Name_of_Seed' in df.columns:
        df.dropna(subset=['Name_of_Seed'], inplace=True)

    # -----------------------------------------
    # remove rows fully empty
    # -----------------------------------------
    df.dropna(how='all', inplace=True)


    # -----------------------------------------
    # Fill null values with statistical Data.
    # -----------------------------------------
    df = fill_categorical_nulls_with_mode(df)
    df = fill_numeric_nulls_with_mean_median(df)
    
    # -----------------------------------------
    # reset index
    # -----------------------------------------
    df.reset_index(drop=True, inplace=True)

    return df

# Perform outlier treatment on the clean scraped data frame

# DATA COLLECTION

In [28]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"

}

In [5]:
all_seeds_data = []

In [6]:
#ALL SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 33): #32

    page_url = f"{base_url}/collections/seeds?page={page}"

    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan

        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )
        if price_tag:

                 # Sale Price
            sale = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            ).get_text(strip=True)
    
            # Regular Price
            regular = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            ).get_text(strip=True)
    
            # remove ₹ and commas
            sale_num = int(re.sub(r"[^\d]", "", sale))/100
    
            regular_num = int(re.sub(r"[^\d]", "", regular))/100
    
            # discount 
            discount = round(
                (regular_num - sale_num),
                2
            )
        
            # discount %
            discount_percent = round(
                ((regular_num - sale_num) / regular_num) * 100,
                2
            )
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        else:
            product_data["Sale_Price"] = np.nan
            product_data["Regular_Price"] = np.nan
            product_data["Discount"] = np.nan
            product_data["Discount_Percent"] = np.nan
            
        # product = {
        #     "sale_price": sale_num,
        #     "regular_price": regular_num,
        #     "discount":discount,
        #     "discount_percent": discount_percent
        # }
        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()
        # print(data)
        all_seeds_data.append(product_data)

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        all_seeds_data.append(product_data)

# FINAL DATAFRAME
# print(all_products)


In [19]:
len(all_seeds_data)

736

In [20]:
all_seeds_df = pd.DataFrame(all_seeds_data)

In [21]:
all_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Water / Soaking Required,Product Ideal For,Product_URL,Blooming Time,
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,"Yes, 24–48 hrs in warm water","Indoor germination, warm well-lit rooms",https://organicbazar.net/products/tomato-seed-...,NaN,NaN
1,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,"Yes, 24–48 hrs in warm water","Indoor germination, warm well-lit rooms",https://organicbazar.net/products/tomato-seed-...,NaN,NaN
2,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/okra-or-lady...,NaN,NaN
3,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/okra-or-lady...,NaN,NaN
4,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,6-12 Days,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/cucumber-see...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,NaN,0.00,95.0,199.0,104.0,52.26,30 Seeds,"Hybrid,","Winter Season,",...,7–14 Days,Above 70%,NaN,"15x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/matthiola-st...,60-80 Days,NaN
732,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/safed-musli-...,NaN,
733,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/safed-musli-...,NaN,
734,Lisianthus Deep Rose Cut Flower Seeds (30 Seed...,NaN,0.00,247.0,349.0,102.0,29.23,30 Seeds,"Open Pollinated,","Winter Season,",...,10 - 21 Days,Above 70%,NaN,"15x15 inch,",Hard - Suitable for experienced gardeners,NaN,NaN,https://organicbazar.net/products/lisianthus-c...,90-120 Days,NaN


In [23]:
all_seeds_df.to_csv('All_Seeds_Df_Raw.csv')

In [27]:
len(all_seeds_data)

954

In [28]:
best_seller_data = []

## BEST SELLERS

In [29]:
#BEST SELLERS PRODUCTS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 41): #40

    page_url = f"{base_url}/collections/best-sellers?page={page}"

    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan

        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##
        # if price_tag:

        #          # Sale Price
        #     sale = price_tag.find(
        #         "span",
        #         class_="price-item price-item--sale price-item--last"
        #     ).get_text(strip=True)
    
        #     # Regular Price
        #     regular = price_tag.find(
        #         "s",
        #         class_="price-item price-item--regular"
        #     ).get_text(strip=True)
    
        #     # remove ₹ and commas
        #     sale_num = int(re.sub(r"[^\d]", "", sale))
    
        #     regular_num = int(re.sub(r"[^\d]", "", regular))
    
        #     # discount 
        #     discount = round(
        #         (regular_num - sale_num),
        #         2
        #     )
        
        #     # discount %
        #     discount_percent = round(
        #         ((regular_num - sale_num) / regular_num) * 100,
        #         2
        #     )
        #     product_data["Sale_Price"] = sale_num
        #     product_data["Regular_Price"] = regular_num
        #     product_data["Discount"] = discount
        #     product_data["Discount_Percent"] = discount_percent
        # else:
        #     product_data["Sale_Price"] = np.nan
        #     product_data["Regular_Price"] = np.nan
        #     product_data["Discount"] = np.nan
        #     product_data["Discount_Percent"] = np.nan
            
        # product = {
        #     "sale_price": sale_num,
        #     "regular_price": regular_num,
        #     "discount":discount,
        #     "discount_percent": discount_percent
        # }
        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()
        # print(data)
        # all_seeds_data.append(product_data)

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        best_seller_data.append(product_data)

# FINAL DATAFRAME
# print(all_products)


In [30]:
best_seller_df = pd.DataFrame(best_seller_data)

In [31]:
best_seller_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Durability,Handles,Special Benefits,Shape,Color,Accessories Type,Fruits,Blooming Time,,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OrganicBazar 12×12 HDPE Round Grow Bag – Premi...,302,4.63,799.0,1990.0,1191.0,59.85,NaN,NaN,NaN,...,5 to 7 years,Without Handles,Foldable,Round,Green,Planter,NaN,NaN,NaN,NaN
3,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN


In [32]:
best_seller_df.to_csv('Best_Seller_Data_Raw.csv')

## ORGANIC SEEDS

In [33]:
organic_seeds_data = []

In [35]:
#ORGANIC SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 2): #1
# https://organicbazar.net/collections/organic-seeds
    page_url = f"{base_url}/collections/organic-seeds"

    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan

        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##
        # if price_tag:

        #          # Sale Price
        #     sale = price_tag.find(
        #         "span",
        #         class_="price-item price-item--sale price-item--last"
        #     ).get_text(strip=True)
    
        #     # Regular Price
        #     regular = price_tag.find(
        #         "s",
        #         class_="price-item price-item--regular"
        #     ).get_text(strip=True)
    
        #     # remove ₹ and commas
        #     sale_num = int(re.sub(r"[^\d]", "", sale))
    
        #     regular_num = int(re.sub(r"[^\d]", "", regular))
    
        #     # discount 
        #     discount = round(
        #         (regular_num - sale_num),
        #         2
        #     )
        
        #     # discount %
        #     discount_percent = round(
        #         ((regular_num - sale_num) / regular_num) * 100,
        #         2
        #     )
        #     product_data["Sale_Price"] = sale_num
        #     product_data["Regular_Price"] = regular_num
        #     product_data["Discount"] = discount
        #     product_data["Discount_Percent"] = discount_percent
        # else:
        #     product_data["Sale_Price"] = np.nan
        #     product_data["Regular_Price"] = np.nan
        #     product_data["Discount"] = np.nan
        #     product_data["Discount_Percent"] = np.nan
            
        # product = {
        #     "sale_price": sale_num,
        #     "regular_price": regular_num,
        #     "discount":discount,
        #     "discount_percent": discount_percent
        # }
        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()
        # print(data)
        # all_seeds_data.append(product_data)

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        organic_seeds_data.append(product_data)

# FINAL DATAFRAME
# print(all_products)


In [37]:
len(organic_seeds_data)

14

In [39]:
len(best_seller_data)

78

In [40]:
organic_seeds_df = pd.DataFrame(organic_seeds_data)

In [41]:
organic_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,
0,Untreated Cherry Tomato Seeds For Organic Home...,24,4.17,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18-30 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cher...,NaN
1,Untreated Okra or Lady Finger Seeds For Organi...,15,4.20,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-okra...,NaN
2,Untreated Spinach Seeds For Organic Gardening ...,26,4.85,69.0,110.0,41.0,37.27,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Above 70%,30–45 Days,"24x15 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-spin...,NaN
3,Untreated Tomato Seeds For Organic Gardening -...,19,4.37,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18–28°C,10- 15 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-toma...,NaN
4,Untreated Broccoli Seeds For Organic Gardening...,9,4.22,49.0,99.0,50.0,50.51,150 Seeds,"Untreated,","Winter Season,",15-26 °C,7–14 Days,Above 80%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-broc...,NaN
5,Untreated Beetroot (Chukandar) seeds For Organ...,17,4.76,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-beet...,NaN
6,Untreated Radish White Long Seeds For Organic ...,15,4.93,30.0,110.0,80.0,72.73,250 Seeds,"Open Pollinated,","Winter Season,",12–24°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-radi...,NaN
7,Untreated Cauliflower Seeds For Organic Garden...,8,4.75,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,90-120 Days,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-caul...,NaN
8,Untreated Coriander (Dhaniya) Seeds For Organi...,6,4.83,69.0,120.0,51.0,42.50,1000 Seeds,"Open Pollinated,",All Season,15-26 °C,7-21 Days,Above 70%,40-45 Days,"24x6 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cori...,NaN
9,Untreated Carrot Orange Seeds For Organic Gard...,5,4.40,59.0,99.0,40.0,40.40,1000 Seeds,"Untreated,","Winter Season,",15-26 °C,7-21 Days,Above 70%,55-70 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-carr...,NaN


In [44]:
organic_seeds_df.add({"Organic":1}*len(organic_seeds_data))

TypeError: unsupported operand type(s) for *: 'dict' and 'int'

In [45]:
organic_seeds_df['Is_Organic'] = 1

In [46]:
organic_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,,Is_Organic
0,Untreated Cherry Tomato Seeds For Organic Home...,24,4.17,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18-30 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cher...,NaN,1
1,Untreated Okra or Lady Finger Seeds For Organi...,15,4.20,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-okra...,NaN,1
2,Untreated Spinach Seeds For Organic Gardening ...,26,4.85,69.0,110.0,41.0,37.27,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Above 70%,30–45 Days,"24x15 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-spin...,NaN,1
3,Untreated Tomato Seeds For Organic Gardening -...,19,4.37,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18–28°C,10- 15 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-toma...,NaN,1
4,Untreated Broccoli Seeds For Organic Gardening...,9,4.22,49.0,99.0,50.0,50.51,150 Seeds,"Untreated,","Winter Season,",15-26 °C,7–14 Days,Above 80%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-broc...,NaN,1
5,Untreated Beetroot (Chukandar) seeds For Organ...,17,4.76,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-beet...,NaN,1
6,Untreated Radish White Long Seeds For Organic ...,15,4.93,30.0,110.0,80.0,72.73,250 Seeds,"Open Pollinated,","Winter Season,",12–24°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-radi...,NaN,1
7,Untreated Cauliflower Seeds For Organic Garden...,8,4.75,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,90-120 Days,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-caul...,NaN,1
8,Untreated Coriander (Dhaniya) Seeds For Organi...,6,4.83,69.0,120.0,51.0,42.50,1000 Seeds,"Open Pollinated,",All Season,15-26 °C,7-21 Days,Above 70%,40-45 Days,"24x6 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cori...,NaN,1
9,Untreated Carrot Orange Seeds For Organic Gard...,5,4.40,59.0,99.0,40.0,40.40,1000 Seeds,"Untreated,","Winter Season,",15-26 °C,7-21 Days,Above 70%,55-70 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-carr...,NaN,1


In [47]:
organic_seeds_df.to_csv('Organic_Seeds_Data_Raw.csv')

In [48]:
organic_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,,Is_Organic
0,Untreated Cherry Tomato Seeds For Organic Home...,24,4.17,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18-30 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cher...,NaN,1
1,Untreated Okra or Lady Finger Seeds For Organi...,15,4.20,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-okra...,NaN,1
2,Untreated Spinach Seeds For Organic Gardening ...,26,4.85,69.0,110.0,41.0,37.27,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Above 70%,30–45 Days,"24x15 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-spin...,NaN,1
3,Untreated Tomato Seeds For Organic Gardening -...,19,4.37,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18–28°C,10- 15 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-toma...,NaN,1
4,Untreated Broccoli Seeds For Organic Gardening...,9,4.22,49.0,99.0,50.0,50.51,150 Seeds,"Untreated,","Winter Season,",15-26 °C,7–14 Days,Above 80%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-broc...,NaN,1
5,Untreated Beetroot (Chukandar) seeds For Organ...,17,4.76,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-beet...,NaN,1
6,Untreated Radish White Long Seeds For Organic ...,15,4.93,30.0,110.0,80.0,72.73,250 Seeds,"Open Pollinated,","Winter Season,",12–24°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-radi...,NaN,1
7,Untreated Cauliflower Seeds For Organic Garden...,8,4.75,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,90-120 Days,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-caul...,NaN,1
8,Untreated Coriander (Dhaniya) Seeds For Organi...,6,4.83,69.0,120.0,51.0,42.50,1000 Seeds,"Open Pollinated,",All Season,15-26 °C,7-21 Days,Above 70%,40-45 Days,"24x6 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cori...,NaN,1
9,Untreated Carrot Orange Seeds For Organic Gard...,5,4.40,59.0,99.0,40.0,40.40,1000 Seeds,"Untreated,","Winter Season,",15-26 °C,7-21 Days,Above 70%,55-70 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-carr...,NaN,1


## Vegetable Seeds

#### Summer Vegetable Seeds

In [50]:
summer_vegetable_seeds_data = []

In [62]:
#SUMMER VEGETABLE SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 11): #10
# https://organicbazar.net/collections/summer-vegetable-seeds?page={page}
    page_url = f"{base_url}/collections/summer-vegetable-seeds?page={page}"

    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan

        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        summer_vegetable_seeds_data.append(product_data)


In [66]:
summer_vegetable_seeds_df = pd.DataFrame(summer_vegetable_seeds_data)

In [67]:
summer_vegetable_seeds_df['Is_Summer'] = 1

In [68]:
summer_vegetable_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Water / Soaking Required,Product Ideal For,Product_URL,Blooming Time,,Is_Summer
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,"Yes, 24–48 hrs in warm water","Indoor germination, warm well-lit rooms",https://organicbazar.net/products/tomato-seed-...,NaN,NaN,1
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/okra-or-lady...,NaN,NaN,1
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/cucumber-see...,NaN,NaN,1
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/coriander-se...,NaN,NaN,1
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/spinach-seeds,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,Chilli Yellowish Light Green (Thick wall fruit...,2,1.00,79.0,149.0,70.0,46.98,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/chilli-yello...,NaN,,1
274,Brinjal Pink Round F1 Hybrid Seeds - (50 Seeds...,1,5.00,35.0,99.0,64.0,64.65,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/brinjal-pink...,NaN,,1
275,Bitter Gourd (Barahmasi) Seeds - 12 Seeds (kar...,5,4.20,59.0,99.0,40.0,40.40,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/bitter-gourd...,NaN,,1
276,Brinjal F1 Long Verde Hybrid Seeds (Baingan/Eg...,5,3.40,40.0,90.0,50.0,55.56,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/brinjal-f1-l...,NaN,,1


In [77]:
summer_vegetable_seeds_df.to_csv('Summer_Vegetable_Seeds_Data_Raw.csv')

In [65]:
len(summer_vegetable_seeds_data)

278

## EXOTIC SEEDS

In [53]:
exotic_seeds_data = []

In [85]:
#EXOTIC VEGETABLE SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 2): #1
# https://organicbazar.net/collections/exotic-vegetables-seeds
    page_url = f"{base_url}/collections/exotic-vegetables-seeds"

    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan

        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        exotic_seeds_data.append(product_data)


In [88]:
exotic_seeds_df = pd.DataFrame(exotic_seeds_data)

In [89]:
exotic_seeds_df['Is_Exotic'] = 1

In [90]:
exotic_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,,Is_Exotic
0,Broccoli Seeds - 150 Seeds (Hari Gobhi/हरी गोभ...,53,4.75,117.0,199.0,82.0,41.21,150 Seeds,"Hybrid,","Winter Season,",15-26 °C,7–14 Days,Above 80%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/broccoli-see...,NaN,1
1,Cherry Tomato Cheramy Seeds - 30 Seeds (Cherry...,18,4.11,148.0,199.0,51.0,25.63,30 Seeds,"Hybrid,",All Season,18-30 °C,7-21 Days,Above 80%,60-80 Days,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/cherry-tomat...,NaN,1
2,Zucchini Black Beauty (Green) Seeds - 15 Seeds...,30,4.63,120.0,160.0,40.0,25.00,15 Seeds,"Hybrid,","Summer Season,",22-30 °C.,7–14 Days,Above 80%,55-70 Days,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/zucchini-bla...,NaN,1
3,Zucchini Yellow Seeds - 15 Seeds (पीली जुकिनी ...,24,4.29,120.0,169.0,49.0,28.99,15 Seeds,"Open Pollinated,","Summer Season,",22-30 °C.,7–14 Days,Above 70%,55-70 Days,"12x12 inch 12x15 inch, 15x15 inch",Easy – Ideal for beginners,https://organicbazar.net/products/zucchini-yellow,NaN,1
4,Brussels Sprouts Seeds - 150 Seeds (ब्रसेल्स स...,14,4.71,49.0,199.0,150.0,75.38,150 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,90-120 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/brussels-spr...,NaN,1
5,Asparagus Seeds - 50 Seeds (Shatavari/शतावरी क...,14,4.79,65.0,90.0,25.0,27.78,50 Seeds,"Open Pollinated,","Rainy Season,",18-30 °C,14-28 Days,Above 70%,NaN,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/asparagus,NaN,1
6,Curly Kale Seeds - 100 Seeds (केल के बीज) Hig...,13,4.46,69.0,149.0,80.0,53.69,100 Seeds,"Open Pollinated,","Winter Season,",18-25°C,7-21 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/curly-kale-s...,NaN,1
7,Kale Seeds For Home Gardening - 150 Seeds (केल...,4,4.75,120.0,169.0,49.0,28.99,150 Seeds,"Hybrid,","Winter Season,",18-25°C,6-12 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/kale-seeds,NaN,1
8,Squash Patty Pan Seeds - 10 Seeds (पैटीपैन स्क...,6,5.00,59.0,110.0,51.0,46.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/squash-zucch...,,1
9,Red kale Seeds - 250 Seeds ( केल के बीज) High ...,11,4.73,49.0,99.0,50.0,50.51,250 Seeds,"Hybrid,","Winter Season,",18-25°C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/red-kale-seeds,NaN,1


In [92]:
len(exotic_seeds_data)

15

In [94]:
exotic_seeds_df.to_csv('Exotic_Seeds_Data_Raw.csv')

## WINTER VEGETABLE SEEDS

In [63]:
winter_seeds_data = []

In [72]:
#WINTER VEGETABLE SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 12): #11
# https://organicbazar.net/collections/winter-vegetable-seeds?page={page}
    page_url = f"{base_url}/collections/winter-vegetable-seeds?page={page}"

    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan

        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        winter_seeds_data.append(product_data)


In [73]:
winter_seeds_df = pd.DataFrame(winter_seeds_data)

In [74]:
winter_seeds_df['Is_Winter'] = 1

In [75]:
winter_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Water / Soaking Required,Product Ideal For,Product_URL,,Is_Winter
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,"Yes, 24–48 hrs in warm water","Indoor germination, warm well-lit rooms",https://organicbazar.net/products/tomato-seed-...,NaN,1
1,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,7-21 Days,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/coriander-se...,NaN,1
2,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,7–14 Days,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/spinach-seeds,NaN,1
3,Brinjal Seeds Black - 50 Seeds) High Germinati...,48,4.15,69.0,99.0,30.0,30.30,50 Seeds,"Open Pollinated,",All Season,...,7–14 Days,Above 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/brinjal-seed...,NaN,1
4,Capsicum (Hari Shimla Mirch) Seeds F1 Hybrid -...,58,4.69,120.0,149.0,29.0,19.46,30 Seeds,F-1 Hybrid,All Season,...,7-21 Days,Above 80%,60-80 Days,"24x12 inch,",Hard - Suitable for experienced gardeners,NaN,NaN,https://organicbazar.net/products/capsicum-see...,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/brinjal-f1-l...,,1
123,Salad Cabbage F1 Hybrid Seeds - 100 Seeds (Pat...,2,4.00,69.0,149.0,80.0,53.69,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/salad-cabbag...,,1
124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/radish-sango...,,1
125,Collard Seeds (300 Seeds),3,5.00,49.0,99.0,50.0,50.51,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/collard-seeds,,1


In [76]:
winter_seeds_df.to_csv('Winter_Seeds_Data_Raw.csv')

## RAINTY SEASON SEEDS

In [27]:
rainy_seeds_data = []

In [29]:
#RAINY VEGETABLE SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 10): #9
# https://organicbazar.net/collections/rainy-season-vegetable-seeds
    page_url = f"{base_url}/collections/rainy-season-vegetable-seeds?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        rainy_seeds_data.append(product_data)


In [30]:
len(rainy_seeds_data)

64

In [31]:
rainy_seeds_df = pd.DataFrame(rainy_seeds_data)

In [32]:
rainy_seeds_df['Is_Rainy'] = 1

In [111]:
rainy_seeds_df.to_csv('Rainy_Seeds_Data_Raw.csv')

## Herb Seeds

In [97]:
herb_seeds_data = []

In [112]:
#HERB SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 3): #2
# https://organicbazar.net/collections/herb-seeds?page={page}
    page_url = f"{base_url}/collections/herb-seeds?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        herb_seeds_data.append(product_data)


In [113]:
len(herb_seeds_data)

62

In [114]:
herb_seeds_df = pd.DataFrame(herb_seeds_data)

In [118]:
herb_seeds_df.to_csv('Herb_Seeds_Data_Raw')

## FLOWER SEEDS

In [119]:
winter_flower_seeds_data = []

In [120]:
#WINTER FLOWER SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 12): #11
# https://organicbazar.net/collections/winter-flower-seed?page={page}
    page_url = f"{base_url}/collections/winter-flower-seed?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        winter_flower_seeds_data.append(product_data)


In [122]:
len(winter_flower_seeds_data)

105

In [157]:
winter_flower_seeds_df = pd.DataFrame(winter_flower_seeds_data)

In [158]:
winter_flower_seeds_df.to_csv('Winter_Flower_Seeds_Data_Raw.csv')

## Summer Flower Seeds

In [121]:
summer_flower_seeds_data = []

In [124]:
#SUMMER FLOWER SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 7): #6
# https://organicbazar.net/collections/summer-flower-seeds?page={page}
    page_url = f"{base_url}/collections/summer-flower-seeds?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        summer_flower_seeds_data.append(product_data)


In [126]:
summer_flower_seeds_df = pd.DataFrame(summer_flower_seeds_data)

In [127]:
summer_flower_seeds_df.to_csv('summer_flower_seeds_data_raw')

In [125]:
len(summer_flower_seeds_data)

80

## Rainy Flower Seeds

In [128]:
rainy_flower_seeds_data = []

In [129]:
#RAINY FLOWER SEEDS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 6): #5
# https://organicbazar.net/collections/rainy-season-flowers-seeds?page=5
    page_url = f"{base_url}/collections/rainy-season-flowers-seeds?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        rainy_flower_seeds_data.append(product_data)


In [130]:
len(rainy_flower_seeds_data)

67

In [131]:
rainy_flower_seeds_df = pd.DataFrame(rainy_flower_seeds_data)

In [132]:
rainy_flower_seeds_df.to_csv('rainy_flower_seeds_data_Raw.csv')

## Bulbs

In [133]:
bulbs_data = []

In [134]:
#BULBS

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 8): #7
#https://organicbazar.net/collections/bulbs?page=7
    page_url = f"{base_url}/collections/bulbs?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        bulbs_data.append(product_data)


In [135]:
len(bulbs_data)

101

In [136]:
bulbs_df = pd.DataFrame(bulbs_data)

In [137]:
bulbs_df.to_csv('bulbs_data_Raw.csv')

## Mircrogreen Seeds

#### https://organicbazar.net/collections/microgreen-seeds

In [141]:
microgreen_seeds_data = []

In [143]:
# Microgreen_seeds_data

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 3): #2
# https://organicbazar.net/collections/microgreen-seeds?page=2
    page_url = f"{base_url}/collections/microgreen-seeds?page={page}"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        microgreen_seeds_data.append(product_data)


In [144]:
len(microgreen_seeds_data)

25

In [155]:
micro_seeds_df = pd.DataFrame(microgreen_seeds_data)
micro_seeds_df.to_csv('micro_seeds_data_Raw.csv')

## Fruit Seeds

In [145]:
fruit_seeds_data = []

In [146]:
# Fruit Seeds DATA

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 2): #1
# https://organicbazar.net/collections/fruit-seeds
    page_url = f"{base_url}/collections/fruit-seeds"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        fruit_seeds_data.append(product_data)


In [147]:
len(fruit_seeds_data)

16

In [150]:
fruit_seeds_df = pd.DataFrame(fruit_seeds_data)
fruit_seeds_df.to_csv('fruit_seeds_data_Raw.csv')

## Other Seeds

In [148]:
other_seeds_data = []

In [149]:
# Other Seeds DATA

base_url = "https://organicbazar.net"

# PAGINATION LOOP
for page in range(1, 2): #1
# https://organicbazar.net/collections/other-seeds
    page_url = f"{base_url}/collections/other-seeds"
    
    response = requests.get(page_url)

    soup = BeautifulSoup(response.text, "html.parser")

    product_links = []

    # EXTRACT PRODUCT LINKS
    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/products/" in href:

            full_link = base_url + href

            if full_link not in product_links:

                product_links.append(full_link)

    # DETAIL PAGE LOOP
    for product_url in product_links:

        response = requests.get(product_url)

        bs_soup_detail = BeautifulSoup(response.text, "html.parser")

        product_data = {}

        # =========================
        # TITLE
        # =========================

        title = bs_soup_detail.find("h1", class_='h2')

        if title:

            a = title.text.strip()

            b = a
            # b = re.findall(
            #     r'(.+) Seeds -|Untreated (.+) seeds -|(\w+)[^Untreated](\w+.+) Seeds -',
            #     a
            # )

            if len(b) > 0:

                # c = ' '.join(b[0]).strip()

                product_data["Name_of_Seed"] = b

            else:

                product_data["Name_of_Seed"] = a

        else:

            product_data["Name_of_Seed"] = np.nan
        # =========================
        # REVIEWS
        # =========================

        review_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__text'
        )

        if review_tag:

            c = review_tag.text.strip()

            d = re.findall(r'\d+', c)

            if len(d) > 0:

                product_data["No_of_Reviews"] = d[0]

            else:

                product_data["No_of_Reviews"] = np.nan

        else:

            product_data["No_of_Reviews"] = np.nan

        # =========================
        # RATING
        # =========================

        rating_tag = bs_soup_detail.find(
            'span',
            class_='jdgm-prev-badge__stars'
        )

        if rating_tag:

            c = rating_tag.get('data-score')

            product_data["Average_Rating"] = c

        else:

            product_data["Average_Rating"] = np.nan


        # =========================
        # REGULAR AND SALE PRICE AND DISCOUNT / DISCOUNT PERCENTAGE
        # =========================

        price_tag = bs_soup_detail.find(
            'div',
            class_='price__sale'
        )

        ##
        if price_tag:

            # Sale Price
            sale_tag = price_tag.find(
                "span",
                class_="price-item price-item--sale price-item--last"
            )
        
            regular_tag = price_tag.find(
                "s",
                class_="price-item price-item--regular"
            )
        
            # ---------------- SALE ----------------
        
            if sale_tag:
        
                sale = sale_tag.get_text(strip=True)
        
                cleaned_sale = re.sub(r"[^\d]", "", sale)
        
                sale_num = int(cleaned_sale)/100 if cleaned_sale else np.nan
        
            else:
        
                sale_num = np.nan
        
            # ---------------- REGULAR ----------------
        
            if regular_tag:
        
                regular = regular_tag.get_text(strip=True)
        
                cleaned_regular = re.sub(r"[^\d]", "", regular)
        
                regular_num = int(cleaned_regular)/100 if cleaned_regular else sale_num
        
            else:
        
                regular_num = sale_num
        
            # ---------------- DISCOUNT ----------------
        
            if pd.notna(sale_num) and pd.notna(regular_num):
        
                discount = round(
                    regular_num - sale_num,
                    2
                )
        
                discount_percent = round(
                    ((regular_num - sale_num) / regular_num) * 100,
                    2
                )
        
            else:
        
                discount = np.nan
                discount_percent = np.nan
        
            # store
        
            product_data["Sale_Price"] = sale_num
            product_data["Regular_Price"] = regular_num
            product_data["Discount"] = discount
            product_data["Discount_Percent"] = discount_percent
        ##

        
        # =========================
        # PRODUCT DETAILS
        # =========================

        product_details_details = bs_soup_detail.find_all('div',class_='products_details-content-inner')

        Text = [product.text.strip() for product in product_details_details]
        data = {}
        text = '\n'.join(Text)
        lines = text.strip().split("\n")
        key = ''
        value = ''
        for line in lines:
    
            if ":" in line:
                # print(line)
                key,value = line.split(":",1)
            else:
                value = line.strip()
            data[key.strip()] = value.strip()
            product_data[key.strip()] = value.strip()

        # =========================
        # PRODUCT URL
        # =========================

        product_data["Product_URL"] = product_url
        
        # APPEND ONE PRODUCT
        other_seeds_data.append(product_data)


In [151]:
len(other_seeds_data)

7

In [152]:
other_seeds_df = pd.DataFrame(other_seeds_data)
other_seeds_df.to_csv('other_seeds_data_Raw.csv')

## COMBINED DATA FRAME

In [160]:
combined_df = pd.concat([
    all_seeds_df,
    other_seeds_df,
    fruit_seeds_df,
    micro_seeds_df,
    bulbs_df,
    rainy_flower_seeds_df,
    summer_flower_seeds_df,
    winter_flower_seeds_df,
    herb_seeds_df,
    summer_vegetable_seeds_df,
    exotic_seeds_df,
    winter_seeds_df,
    rainy_seeds_df,
    organic_seeds_df,
    best_seller_df
], ignore_index=True)

In [161]:
combined_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [162]:
combined_df.drop_duplicates()

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [163]:
combined_df = (
    combined_df
    .dropna(how="all")
    .reset_index(drop=True)
)

In [170]:
cd = combined_df.drop(columns={'Product_URL'})

In [171]:
cd.dropna(how='all')

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1848,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [173]:
cd.iloc[1848].isna().sum()

np.int64(40)

In [174]:
cd.columns

Index(['Name_of_Seed', 'No_of_Reviews', 'Average_Rating', 'Sale_Price',
       'Regular_Price', 'Discount', 'Discount_Percent', 'Number of Seeds',
       'Seed Type', 'Sowing Season', 'Germination Temperature',
       'Germination Time', 'Germination Rate', 'First Harvest',
       'Container Requirement / Pots', 'Growing Level',
       'Water / Soaking Required', 'Product Ideal For', 'Blooming Time', '',
       'Accessories Type', 'Plant Size', 'Is_Summer', 'Is_Exotic', 'Is_Winter',
       'Is_Rainy', 'Is_Organic', 'Material', 'GSM', 'Plant Type Suitability',
       'Vegetables', 'Flowers', 'Drainage', 'Capacity', 'Durability',
       'Handles', 'Special Benefits', 'Shape', 'Color', 'Fruits', 'Herbs'],
      dtype='object')

In [182]:
final_df = cd.dropna(subset=["Name_of_Seed"])

In [187]:
final_df = final_df.drop_duplicates().reset_index()

In [188]:
final_df

,index,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
0,0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841,1816,All Time Vegetable Seeds Kit for Home Gardening,113,4.46,199.0,450.0,251.0,55.78,NaN,"Hybrid,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
842,1818,Neem Cake Powder (Neem khali) for Plants,72,4.85,199.0,399.0,200.0,50.13,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
843,1825,HDPE 18x18 Grow Bags for Terrace Gardening Ext...,119,4.76,1049.0,1495.0,446.0,29.83,NaN,NaN,...,NaN,Drainage Holes,75.06 L,5 to 7 years,Without Handles,Foldable,Round,Green,"Strawberries, Pineapple, Dwarf Lemon, Dwarf Pa...",NaN
844,1826,Organic Steamed Bone Meal Fertilizer for Plant...,46,4.72,499.0,999.0,500.0,50.05,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [189]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 846 entries, 0 to 845
Data columns (total 42 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   index                         846 non-null    int64  
 1   Name_of_Seed                  846 non-null    object 
 2   No_of_Reviews                 750 non-null    object 
 3   Average_Rating                846 non-null    object 
 4   Sale_Price                    846 non-null    float64
 5   Regular_Price                 846 non-null    float64
 6   Discount                      846 non-null    float64
 7   Discount_Percent              846 non-null    float64
 8   Number of Seeds               700 non-null    object 
 9   Seed Type                     665 non-null    object 
 10  Sowing Season                 727 non-null    object 
 11  Germination Temperature       727 non-null    object 
 12  Germination Time              727 non-null    object 
 13  Germi

In [190]:
final_df.describe()

,index,Sale_Price,Regular_Price,Discount,Discount_Percent,Is_Summer,Is_Exotic,Is_Winter,Is_Rainy,Is_Organic
count,846.000000,846.000000,846.000000,846.000000,846.000000,157.0,15.0,110.0,56.0,14.0
mean,1015.130024,124.381797,212.865248,88.483452,43.105402,1.0,1.0,1.0,1.0,1.0
std,544.604571,149.323467,248.606270,111.366367,12.861514,0.0,0.0,0.0,0.0,0.0
min,0.000000,22.000000,70.000000,1.000000,1.010000,1.0,1.0,1.0,1.0,1.0
25%,584.500000,49.000000,99.000000,40.000000,33.560000,1.0,1.0,1.0,1.0,1.0
50%,972.500000,69.000000,110.000000,50.000000,40.400000,1.0,1.0,1.0,1.0,1.0
75%,1559.750000,120.000000,199.000000,90.000000,50.510000,1.0,1.0,1.0,1.0,1.0
max,1843.000000,1699.000000,2999.000000,1300.000000,80.400000,1.0,1.0,1.0,1.0,1.0


In [191]:
final_df = final_df.drop(columns={'index'})

In [193]:
final_df = final_df.drop(columns={''})

In [194]:
final_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841,All Time Vegetable Seeds Kit for Home Gardening,113,4.46,199.0,450.0,251.0,55.78,NaN,"Hybrid,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
842,Neem Cake Powder (Neem khali) for Plants,72,4.85,199.0,399.0,200.0,50.13,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
843,HDPE 18x18 Grow Bags for Terrace Gardening Ext...,119,4.76,1049.0,1495.0,446.0,29.83,NaN,NaN,NaN,...,NaN,Drainage Holes,75.06 L,5 to 7 years,Without Handles,Foldable,Round,Green,"Strawberries, Pineapple, Dwarf Lemon, Dwarf Pa...",NaN
844,Organic Steamed Bone Meal Fertilizer for Plant...,46,4.72,499.0,999.0,500.0,50.05,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [195]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 846 entries, 0 to 845
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  846 non-null    object 
 1   No_of_Reviews                 750 non-null    object 
 2   Average_Rating                846 non-null    object 
 3   Sale_Price                    846 non-null    float64
 4   Regular_Price                 846 non-null    float64
 5   Discount                      846 non-null    float64
 6   Discount_Percent              846 non-null    float64
 7   Number of Seeds               700 non-null    object 
 8   Seed Type                     665 non-null    object 
 9   Sowing Season                 727 non-null    object 
 10  Germination Temperature       727 non-null    object 
 11  Germination Time              727 non-null    object 
 12  Germination Rate              727 non-null    object 
 13  First

In [198]:
final_df['Seed Type'].value_counts()

Seed Type
Open Pollinated,    350
Hybrid,             150
F-1 Hybrid          125
Untreated,           40
Name: count, dtype: int64

In [199]:
organic_seeds_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,,Is_Organic
0,Untreated Cherry Tomato Seeds For Organic Home...,24,4.17,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18-30 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cher...,NaN,1
1,Untreated Okra or Lady Finger Seeds For Organi...,15,4.20,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-okra...,NaN,1
2,Untreated Spinach Seeds For Organic Gardening ...,26,4.85,69.0,110.0,41.0,37.27,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Above 70%,30–45 Days,"24x15 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-spin...,NaN,1
3,Untreated Tomato Seeds For Organic Gardening -...,19,4.37,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,18–28°C,10- 15 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-toma...,NaN,1
4,Untreated Broccoli Seeds For Organic Gardening...,9,4.22,49.0,99.0,50.0,50.51,150 Seeds,"Untreated,","Winter Season,",15-26 °C,7–14 Days,Above 80%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-broc...,NaN,1
5,Untreated Beetroot (Chukandar) seeds For Organ...,17,4.76,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,60-90 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-beet...,NaN,1
6,Untreated Radish White Long Seeds For Organic ...,15,4.93,30.0,110.0,80.0,72.73,250 Seeds,"Open Pollinated,","Winter Season,",12–24°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-radi...,NaN,1
7,Untreated Cauliflower Seeds For Organic Garden...,8,4.75,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",15-26 °C,7–14 Days,Above 70%,90-120 Days,"24x18 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-caul...,NaN,1
8,Untreated Coriander (Dhaniya) Seeds For Organi...,6,4.83,69.0,120.0,51.0,42.50,1000 Seeds,"Open Pollinated,",All Season,15-26 °C,7-21 Days,Above 70%,40-45 Days,"24x6 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-cori...,NaN,1
9,Untreated Carrot Orange Seeds For Organic Gard...,5,4.40,59.0,99.0,40.0,40.40,1000 Seeds,"Untreated,","Winter Season,",15-26 °C,7-21 Days,Above 70%,55-70 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/organic-carr...,NaN,1


In [205]:
final_df[final_df['Is_Organic'] == 1]

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
824,Untreated Cherry Tomato Seeds For Organic Home...,24,4.17,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
825,Untreated Okra or Lady Finger Seeds For Organi...,15,4.20,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
826,Untreated Spinach Seeds For Organic Gardening ...,26,4.85,69.0,110.0,41.0,37.27,750 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
827,Untreated Tomato Seeds For Organic Gardening -...,19,4.37,49.0,99.0,50.0,50.51,50 Seeds,"Untreated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
828,Untreated Broccoli Seeds For Organic Gardening...,9,4.22,49.0,99.0,50.0,50.51,150 Seeds,"Untreated,","Winter Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
829,Untreated Beetroot (Chukandar) seeds For Organ...,17,4.76,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
830,Untreated Radish White Long Seeds For Organic ...,15,4.93,30.0,110.0,80.0,72.73,250 Seeds,"Open Pollinated,","Winter Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
831,Untreated Cauliflower Seeds For Organic Garden...,8,4.75,49.0,99.0,50.0,50.51,200 Seeds,"Open Pollinated,","Winter Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
832,Untreated Coriander (Dhaniya) Seeds For Organi...,6,4.83,69.0,120.0,51.0,42.50,1000 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
833,Untreated Carrot Orange Seeds For Organic Gard...,5,4.40,59.0,99.0,40.0,40.40,1000 Seeds,"Untreated,","Winter Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [206]:
final_df

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Flowers,Drainage,Capacity,Durability,Handles,Special Benefits,Shape,Color,Fruits,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
841,All Time Vegetable Seeds Kit for Home Gardening,113,4.46,199.0,450.0,251.0,55.78,NaN,"Hybrid,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
842,Neem Cake Powder (Neem khali) for Plants,72,4.85,199.0,399.0,200.0,50.13,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
843,HDPE 18x18 Grow Bags for Terrace Gardening Ext...,119,4.76,1049.0,1495.0,446.0,29.83,NaN,NaN,NaN,...,NaN,Drainage Holes,75.06 L,5 to 7 years,Without Handles,Foldable,Round,Green,"Strawberries, Pineapple, Dwarf Lemon, Dwarf Pa...",NaN
844,Organic Steamed Bone Meal Fertilizer for Plant...,46,4.72,499.0,999.0,500.0,50.05,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [207]:
final_df.to_csv('Final_DataFrame_Raw.csv')

# DATA CLEANING

In [212]:
# DATAFRAME1:
all_seeds_df_copy = all_seeds_df

In [215]:
all_seeds_df_copy.drop_duplicates(inplace=True)

In [216]:
all_seeds_df_copy.columns

Index(['Name_of_Seed', 'No_of_Reviews', 'Average_Rating', 'Sale_Price',
       'Regular_Price', 'Discount', 'Discount_Percent', 'Number of Seeds',
       'Seed Type', 'Sowing Season', 'Germination Temperature',
       'Germination Time', 'Germination Rate', 'First Harvest',
       'Container Requirement / Pots', 'Growing Level',
       'Water / Soaking Required', 'Product Ideal For', 'Product_URL',
       'Blooming Time', ''],
      dtype='object')

In [217]:
all_seeds_df_copy.drop(columns={''},inplace=True)

In [221]:
all_seeds_df_copy.reset_index(inplace = True)

In [223]:
all_seeds_df_copy.drop(columns={'index'},inplace=True)

In [231]:
all_seeds_df_copy[all_seeds_df_copy['Name_of_Seed'].isna()]

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Water / Soaking Required,Product Ideal For,Product_URL,Blooming Time
56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/sem-phali-li...,NaN
57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/red-amaranth...,NaN
58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/all-season-f...,NaN
60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/cherry-tomat...,NaN
62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/brinjal-purp...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/beetroot-whi...,NaN
345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/gomphrena-ex...,NaN
347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/clover-micro...,NaN
358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/french-marig...,NaN


In [260]:
df_all_seeds_copy = all_seeds_df_copy.dropna(subset=["Name_of_Seed"])

In [261]:
df_all_seeds_copy

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Water / Soaking Required,Product Ideal For,Product_URL,Blooming Time
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,18–28°C,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,"Yes, 24–48 hrs in warm water","Indoor germination, warm well-lit rooms",https://organicbazar.net/products/tomato-seed-...,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/okra-or-lady...,NaN
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",18-30 °C,6-12 Days,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/cucumber-see...,NaN
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,18–28°C,7-21 Days,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/coriander-se...,NaN
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/spinach-seeds,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362,Salvia Vista Purple Flower Seeds (20 Seeds) Hi...,NaN,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Summer Season,",18–28°C,7-21 Days,Above 70%,NaN,"18x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/salvia-vista...,60-80 Days
363,Knol Khol Purple Microgreen Seeds (25g),NaN,0.00,79.0,149.0,70.0,46.98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/knol-khol-pu...,NaN
365,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,NaN,0.00,95.0,199.0,104.0,52.26,30 Seeds,"Hybrid,","Winter Season,",15-26 °C,7–14 Days,Above 70%,NaN,"15x12 inch,",Easy – Ideal for beginners,NaN,NaN,https://organicbazar.net/products/matthiola-st...,60-80 Days
366,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/safed-musli-...,NaN


In [262]:
df_best_seller_copy = drop_duplicate_from_dataframe(best_seller_df)

In [263]:
    df_best_seller_copy

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Capacity,Durability,Handles,Special Benefits,Shape,Color,Accessories Type,Fruits,Blooming Time,Herbs
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OrganicBazar 12×12 HDPE Round Grow Bag – Premi...,302,4.63,799.0,1990.0,1191.0,59.85,NaN,NaN,NaN,...,22.24 L,5 to 7 years,Without Handles,Foldable,Round,Green,Planter,NaN,NaN,NaN
3,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Brinjal Seeds Black - 50 Seeds) High Germinati...,48,4.15,69.0,99.0,30.0,30.30,50 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Capsicum (Hari Shimla Mirch) Seeds F1 Hybrid -...,58,4.69,120.0,149.0,29.0,19.46,30 Seeds,F-1 Hybrid,All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,OrganicBazar 15x15 Grow Bag for Vegetable Gard...,140,4.61,699.0,999.0,300.0,30.03,NaN,NaN,NaN,...,43.44 L,5 to 7 years,Without Handles,Foldable,Round,Green,Planter,"Strawberries, Pineapple, Dwarf Lemon, Dwarf Pa...",NaN,NaN
9,French Beans Seeds - 20 Seeds (फ्रेंच बीन्स के...,37,4.32,69.0,99.0,30.0,30.30,20 Seeds,"Open Pollinated,",All Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
rainy_seeds_df

NameError: name 'rainy_seeds_df' is not defined

In [264]:
df_organic_seeds_copy = drop_duplicate_from_dataframe(organic_seeds_df)

In [265]:
df_rainy_seeds_copy = drop_duplicate_from_dataframe(rainy_seeds_df)

In [266]:
df_winter_seeds_copy = drop_duplicate_from_dataframe(winter_seeds_df)

In [267]:
df_exotic_seeds_copy = drop_duplicate_from_dataframe(exotic_seeds_df)

In [268]:
df_summer_vegetable_seeds_copy = drop_duplicate_from_dataframe(summer_vegetable_seeds_df)

In [269]:
df_herb_seeds_copy = drop_duplicate_from_dataframe(herb_seeds_df)

In [270]:
df_winter_flower_seeds_copy = drop_duplicate_from_dataframe(winter_flower_seeds_df)

In [271]:
df_summer_flower_seeds_copy = drop_duplicate_from_dataframe(summer_flower_seeds_df)

In [272]:
df_rainy_flower_seeds_copy = drop_duplicate_from_dataframe(rainy_flower_seeds_df)

In [273]:
df_bulbs_copy = drop_duplicate_from_dataframe(bulbs_df)

In [274]:
df_micro_seeds_copy = drop_duplicate_from_dataframe(micro_seeds_df)

In [275]:
df_fruit_seeds_copy = drop_duplicate_from_dataframe(fruit_seeds_df)

In [276]:
df_other_seeds_copy = drop_duplicate_from_dataframe(other_seeds_df)

In [278]:
df_all_seeds_copy[['Name_of_Seed','Product Ideal For']]

,Name_of_Seed,Product Ideal For
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,"Indoor germination, warm well-lit rooms"
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,NaN
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,NaN
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,NaN
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,NaN
...,...,...
362,Salvia Vista Purple Flower Seeds (20 Seeds) Hi...,NaN
363,Knol Khol Purple Microgreen Seeds (25g),NaN
365,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,NaN
366,Safed Musli Bulbs Seeds,NaN


In [279]:
len(df_all_seeds_copy)

275

In [284]:
df_all_seeds_copy['Name_of_Seed'].value_counts()

Name_of_Seed
Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (टमाटर के बीज) High Germination/ Easy to Grow/ Ideal for Terrace/Kitchen Gardening                           1
Mustard (Sarson) Seeds (सरसों के बीज) - 1000 Seeds                                                                                                       1
Snake Gourd Extra long Seeds - 10 Seeds (Chichinda/चिचिंडा के बीज) Easy To grow, High Germination, High Yield Snake Gourd Seeds for Terrace Gardening    1
Water Cress Seeds (500 Seeds)                                                                                                                            1
Vinca Tattoo Blueberry Seeds - 10 Seeds (Periwinkle/ Sadabahar)                                                                                          1
                                                                                                                                                        ..
Chinese Cabbage Seeds - 150 Seeds (Chinese Patta Gobhi/ची

In [282]:
missing_all_seeds = missing_value_percentage(df_all_seeds_copy,df_all_seeds_copy.columns)

In [304]:
missing_cols_of_all_seeds = missing_all_seeds[missing_all_seeds['Missing_Percentage']>95]['Column_Name'].tolist()

In [301]:
missing_cols = missing_all_seeds[
    missing_all_seeds['Missing_Percentage'] > 95
]['Column_Name'].tolist()

In [305]:
drop_column_with_missing_values(df_all_seeds_copy,missing_cols_of_all_seeds)

In [306]:
df_all_seeds_copy

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,Blooming Time
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,18–28°C,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/tomato-seed-...,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/okra-or-lady...,NaN
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",18-30 °C,6-12 Days,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,https://organicbazar.net/products/cucumber-see...,NaN
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,18–28°C,7-21 Days,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/coriander-se...,NaN
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/spinach-seeds,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362,Salvia Vista Purple Flower Seeds (20 Seeds) Hi...,NaN,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Summer Season,",18–28°C,7-21 Days,Above 70%,NaN,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,60-80 Days
363,Knol Khol Purple Microgreen Seeds (25g),NaN,0.00,79.0,149.0,70.0,46.98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/knol-khol-pu...,NaN
365,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,NaN,0.00,95.0,199.0,104.0,52.26,30 Seeds,"Hybrid,","Winter Season,",15-26 °C,7–14 Days,Above 70%,NaN,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/matthiola-st...,60-80 Days
366,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/safed-musli-...,NaN


In [313]:
df_all_seeds_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 275 entries, 0 to 367
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  275 non-null    object 
 1   No_of_Reviews                 249 non-null    object 
 2   Average_Rating                275 non-null    object 
 3   Sale_Price                    275 non-null    float64
 4   Regular_Price                 275 non-null    float64
 5   Discount                      275 non-null    float64
 6   Discount_Percent              275 non-null    float64
 7   Number of Seeds               213 non-null    object 
 8   Seed Type                     219 non-null    object 
 9   Sowing Season                 226 non-null    object 
 10  Germination Temperature       226 non-null    object 
 11  Germination Time              226 non-null    object 
 12  Germination Rate              226 non-null    object 
 13  First Harv

In [315]:
df_all_seeds_copy['No_of_Reviews'] = df_all_seeds_copy['No_of_Reviews'].fillna(0)

In [331]:
df_all_seeds_copy[df_all_seeds_copy['Number of Seeds'].isna()][['Name_of_Seed']][:23]['Name_of_Seed'][:].tolist()

['45 Variety of Vegetable Seeds kit With High Germination Rate',
 '40 Variety of Flowers Seeds kit With High Germination Rate',
 'Summer Vegetable Seeds Kit (20 in 1)',
 'Winter Season Vegetables Seeds Kit – 20 Best Winter Vegetable Seeds for Terrace Gardening 🌱❄️',
 'Rainy Season Vegetable Seeds Kit – Pack of 20 Best Vegetable Seeds for Monsoon Gardening | Ideal for Terrace & Kitchen Garden',
 'Set of 20 Winter Season Flower Seeds Kit',
 'Original Black Turmeric (Kali Haldi ke beej) For Planting (400g)',
 'Winter Vegetable and Flower Mixed Seeds Kit - 20 Best Winter Vegetable And Flowers Seeds for Terrace Gardening 🌱❄️',
 'Extra Long Bottle Gourd (Lauki / Ghiya) Seeds (6 फीट लंबी लौकी के बीज)',
 'Krishna/Shyam Basil Seeds - 200 Seeds (Tulsi/तुलसी के बीज) High Germination/Perfect for pots, balconies, or Terrace gardens',
 '6 Varieties of Leafy Vegetable Seeds Combo Pack',
 'Rainy Season Vegetable & Flower Mixed Seeds Kit – 20 Best Varieties for Terrace & Kitchen Gardening',
 'Untreated

In [332]:
df_all_seeds_copy

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,Blooming Time
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,18–28°C,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/tomato-seed-...,NaN
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/okra-or-lady...,NaN
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",18-30 °C,6-12 Days,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,https://organicbazar.net/products/cucumber-see...,NaN
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,18–28°C,7-21 Days,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/coriander-se...,NaN
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/spinach-seeds,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362,Salvia Vista Purple Flower Seeds (20 Seeds) Hi...,0,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Summer Season,",18–28°C,7-21 Days,Above 70%,NaN,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,60-80 Days
363,Knol Khol Purple Microgreen Seeds (25g),0,0.00,79.0,149.0,70.0,46.98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/knol-khol-pu...,NaN
365,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,0,0.00,95.0,199.0,104.0,52.26,30 Seeds,"Hybrid,","Winter Season,",15-26 °C,7–14 Days,Above 70%,NaN,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/matthiola-st...,60-80 Days
366,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://organicbazar.net/products/safed-musli-...,NaN


In [333]:
df_bulbs_copy

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Product_URL,Number of Seeds,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Seed Type,First Harvest,Accessories Type
0,Original Black Turmeric (Kali Haldi ke beej) F...,20,4.65,299.0,799.0,500.0,62.58,https://organicbazar.net/products/black-turmer...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ranunculus Flower Bulbs Mixed Colors (5N),23,3.87,399.0,599.0,200.0,33.39,https://organicbazar.net/products/ranunculus-f...,05 Bulbs,"Winter Season,",10-15°C.,16-30 Days,Above 70%,90-120 Days,"15x15 inch,",Hard - Suitable for experienced gardeners,NaN,NaN,NaN
2,Rain Lily Flower Bulbs Mixed Color (10 Bulbs) ...,22,4.32,169.0,199.0,30.0,15.08,https://organicbazar.net/products/rain-lily-fl...,10 Bulbs,"Rainy Season,",22-30 °C.,14-28 Days,Above 70%,60-90 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",NaN,NaN
3,Kashmiri Saffron (Kesar) Bulbs 10N,23,4.48,499.0,599.0,100.0,16.69,https://organicbazar.net/products/kashmiri-saf...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Gladiolus Mix Color Flower Bulbs (10N),21,4.71,199.0,299.0,100.0,33.44,https://organicbazar.net/products/gladiolus-fl...,10 Bulbs,"Winter Season,",12–18°C,14-28 Days,Above 70%,60-90 Days,"12x12 inch,",Easy – Ideal for beginners,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,Tulip Doberman (Black Purple) Flower Bulbs,NaN,0.00,450.0,999.0,549.0,54.95,https://organicbazar.net/products/tulip-doberm...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,https://organicbazar.net/products/safed-musli-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,Gladiolus Yellow Flower Bulbs,NaN,0.00,99.0,199.0,100.0,50.25,https://organicbazar.net/products/gladiolus-ye...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,Asiatic (Lilium) Lily Yellow Flower Bulbs,NaN,0.00,299.0,399.0,100.0,25.06,https://organicbazar.net/products/asiatic-lili...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 21 MAY 2026

In [334]:

dataframes = {
    "df_all_seeds_copy": df_all_seeds_copy,
    "df_best_seller_copy": df_best_seller_copy,
    "df_organic_seeds_copy": df_organic_seeds_copy,
    "df_rainy_seeds_copy": df_rainy_seeds_copy,
    "df_winter_seeds_copy": df_winter_seeds_copy,
    "df_exotic_seeds_copy": df_exotic_seeds_copy,
    "df_summer_vegetable_seeds_copy": df_summer_vegetable_seeds_copy,
    "df_herb_seeds_copy": df_herb_seeds_copy,
    "df_winter_flower_seeds_copy": df_winter_flower_seeds_copy,
    "df_summer_flower_seeds_copy": df_summer_flower_seeds_copy,
    "df_rainy_flower_seeds_copy": df_rainy_flower_seeds_copy,
    "df_bulbs_copy": df_bulbs_copy,
    "df_micro_seeds_copy": df_micro_seeds_copy,
    "df_fruit_seeds_copy": df_fruit_seeds_copy,
    "df_other_seeds_copy": df_other_seeds_copy
}

for name, dataframe in dataframes.items():

    dataframe.to_csv(
        f"{name}_uncleaned.csv",
        index=False
    )

print("All CSV files saved successfully.")

All CSV files saved successfully.


# INFO AND DESCRIPTION OF ALL DATA FRAMES

## BEFORE CLEANING

In [341]:
df_all_seeds_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 275 entries, 0 to 367
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  275 non-null    object 
 1   No_of_Reviews                 275 non-null    object 
 2   Average_Rating                275 non-null    object 
 3   Sale_Price                    275 non-null    float64
 4   Regular_Price                 275 non-null    float64
 5   Discount                      275 non-null    float64
 6   Discount_Percent              275 non-null    float64
 7   Number of Seeds               213 non-null    object 
 8   Seed Type                     219 non-null    object 
 9   Sowing Season                 226 non-null    object 
 10  Germination Temperature       226 non-null    object 
 11  Germination Time              226 non-null    object 
 12  Germination Rate              226 non-null    object 
 13  First Harv

In [342]:
df_all_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent
count,275.000000,275.000000,275.000000,275.000000
mean,102.789091,192.589091,89.800000,45.828655
std,141.952492,266.347795,132.503298,13.516602
min,22.000000,70.000000,1.000000,1.010000
25%,49.000000,99.000000,40.000000,34.670000
50%,59.000000,99.000000,51.000000,46.360000
75%,99.000000,174.000000,80.000000,55.560000
max,1699.000000,2999.000000,1300.000000,80.400000


In [345]:
df_all_seeds_copy.isna().sum()

Name_of_Seed                      0
No_of_Reviews                     0
Average_Rating                    0
Sale_Price                        0
Regular_Price                     0
Discount                          0
Discount_Percent                  0
Number of Seeds                  62
Seed Type                        56
Sowing Season                    49
Germination Temperature          49
Germination Time                 49
Germination Rate                 49
First Harvest                   128
Container Requirement / Pots     67
Growing Level                    49
Product_URL                       0
Blooming Time                   196
dtype: int64

In [343]:
df_best_seller_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  56 non-null     object 
 1   No_of_Reviews                 56 non-null     object 
 2   Average_Rating                56 non-null     object 
 3   Sale_Price                    56 non-null     float64
 4   Regular_Price                 56 non-null     float64
 5   Discount                      56 non-null     float64
 6   Discount_Percent              56 non-null     float64
 7   Number of Seeds               45 non-null     object 
 8   Seed Type                     49 non-null     object 
 9   Sowing Season                 49 non-null     object 
 10  Germination Temperature       49 non-null     object 
 11  Germination Time              49 non-null     object 
 12  Germination Rate              49 non-null     object 
 13  First H

In [344]:
df_best_seller_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent
count,56.000000,56.000000,56.000000,56.000000
mean,161.946429,291.035714,129.089286,39.342143
std,222.174904,426.769726,222.073557,11.003964
min,39.000000,90.000000,29.000000,19.460000
25%,59.000000,99.000000,34.000000,30.300000
50%,69.000000,99.000000,42.500000,35.175000
75%,117.750000,179.000000,62.750000,44.907500
max,1049.000000,1990.000000,1191.000000,69.800000


In [346]:
df_best_seller_copy.isna().sum()

Name_of_Seed                     0
No_of_Reviews                    0
Average_Rating                   0
Sale_Price                       0
Regular_Price                    0
Discount                         0
Discount_Percent                 0
Number of Seeds                 11
Seed Type                        7
Sowing Season                    7
Germination Temperature          7
Germination Time                 7
Germination Rate                 7
First Harvest                   21
Container Requirement / Pots     7
Growing Level                    7
Water / Soaking Required        55
Product Ideal For               51
Product_URL                      0
Material                        51
GSM                             52
Plant Type Suitability          52
Vegetables                      53
Flowers                         53
Drainage                        52
Capacity                        52
Durability                      52
Handles                         52
Special Benefits    

In [347]:
df_bulbs_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  92 non-null     object 
 1   No_of_Reviews                 61 non-null     object 
 2   Average_Rating                92 non-null     object 
 3   Sale_Price                    92 non-null     float64
 4   Regular_Price                 92 non-null     float64
 5   Discount                      92 non-null     float64
 6   Discount_Percent              92 non-null     float64
 7   Product_URL                   92 non-null     object 
 8   Number of Seeds               68 non-null     object 
 9   Sowing Season                 68 non-null     object 
 10  Germination Temperature       68 non-null     object 
 11  Germination Time              68 non-null     object 
 12  Germination Rate              68 non-null     object 
 13  Bloomin

In [348]:
df_bulbs_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent
count,92.000000,92.000000,92.000000,92.000000
mean,329.228261,524.000000,194.771739,37.201739
std,168.621633,253.925229,109.030503,9.384569
min,99.000000,149.000000,30.000000,15.080000
25%,199.000000,299.000000,150.000000,30.060000
50%,299.000000,499.000000,150.000000,37.550000
75%,399.000000,599.000000,200.000000,40.080000
max,699.000000,999.000000,549.000000,62.580000


In [349]:
df_bulbs_copy.isna().sum()

Name_of_Seed                     0
No_of_Reviews                   31
Average_Rating                   0
Sale_Price                       0
Regular_Price                    0
Discount                         0
Discount_Percent                 0
Product_URL                      0
Number of Seeds                 24
Sowing Season                   24
Germination Temperature         24
Germination Time                24
Germination Rate                24
Blooming Time                   27
Container Requirement / Pots    24
Growing Level                   24
Seed Type                       83
First Harvest                   89
Accessories Type                91
dtype: int64

In [350]:
df_exotic_seeds_copy.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  15 non-null     object 
 1   No_of_Reviews                 15 non-null     object 
 2   Average_Rating                15 non-null     object 
 3   Sale_Price                    15 non-null     float64
 4   Regular_Price                 15 non-null     float64
 5   Discount                      15 non-null     float64
 6   Discount_Percent              15 non-null     float64
 7   Number of Seeds               12 non-null     object 
 8   Seed Type                     12 non-null     object 
 9   Sowing Season                 12 non-null     object 
 10  Germination Temperature       12 non-null     object 
 11  Germination Time              12 non-null     object 
 12  Germination Rate              12 non-null     object 
 13  First H

In [351]:
df_exotic_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent,Is_Exotic
count,15.000000,15.000000,15.000000,15.000000,15.0
mean,89.533333,150.600000,61.066667,40.746667,1.0
std,34.165285,39.879819,31.210041,15.893285,0.0
min,30.000000,90.000000,25.000000,25.000000,1.0
25%,62.000000,119.500000,44.500000,28.385000,1.0
50%,99.000000,149.000000,50.000000,33.560000,1.0
75%,118.500000,184.000000,70.000000,50.380000,1.0
max,148.000000,199.000000,150.000000,75.380000,1.0


In [352]:
df_exotic_seeds_copy.isna().sum()

Name_of_Seed                    0
No_of_Reviews                   0
Average_Rating                  0
Sale_Price                      0
Regular_Price                   0
Discount                        0
Discount_Percent                0
Number of Seeds                 3
Seed Type                       3
Sowing Season                   3
Germination Temperature         3
Germination Time                3
Germination Rate                3
First Harvest                   4
Container Requirement / Pots    3
Growing Level                   3
Product_URL                     0
Is_Exotic                       0
dtype: int64

In [353]:
df_fruit_seeds_copy.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  16 non-null     object 
 1   No_of_Reviews                 15 non-null     object 
 2   Average_Rating                16 non-null     object 
 3   Sale_Price                    16 non-null     float64
 4   Regular_Price                 16 non-null     float64
 5   Discount                      16 non-null     float64
 6   Discount_Percent              16 non-null     float64
 7   Number of Seeds               16 non-null     object 
 8   Seed Type                     16 non-null     object 
 9   Sowing Season                 16 non-null     object 
 10  Germination Temperature       16 non-null     object 
 11  Germination Time              16 non-null     object 
 12  Germination Rate              16 non-null     object 
 13  First H

In [354]:
df_fruit_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent
count,16.000000,16.000000,16.000000,16.000000
mean,86.125000,140.000000,53.875000,38.536250
std,30.508742,41.992063,24.146428,10.518588
min,49.000000,90.000000,30.000000,20.000000
25%,63.500000,99.000000,38.750000,34.170000
50%,84.000000,120.000000,45.500000,39.580000
75%,99.000000,184.000000,62.500000,46.080000
max,159.000000,199.000000,109.000000,54.770000


In [355]:

df_fruit_seeds_copy.isna().sum()

Name_of_Seed                    0
No_of_Reviews                   1
Average_Rating                  0
Sale_Price                      0
Regular_Price                   0
Discount                        0
Discount_Percent                0
Number of Seeds                 0
Seed Type                       0
Sowing Season                   0
Germination Temperature         0
Germination Time                0
Germination Rate                0
First Harvest                   0
Container Requirement / Pots    1
Growing Level                   0
Product_URL                     0
dtype: int64

In [356]:
df_herb_seeds_copy.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  31 non-null     object 
 1   No_of_Reviews                 26 non-null     object 
 2   Average_Rating                31 non-null     object 
 3   Sale_Price                    31 non-null     float64
 4   Regular_Price                 31 non-null     float64
 5   Discount                      31 non-null     float64
 6   Discount_Percent              31 non-null     float64
 7   Number of Seeds               23 non-null     object 
 8   Seed Type                     22 non-null     object 
 9   Sowing Season                 23 non-null     object 
 10  Germination Temperature       23 non-null     object 
 11  Germination Time              23 non-null     object 
 12  Germination Rate              23 non-null     object 
 13  First H

In [357]:
df_herb_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent
count,31.000000,31.000000,31.000000,31.000000
mean,53.000000,119.225806,66.225806,53.205484
std,20.195709,45.820454,38.812979,13.355655
min,22.000000,89.000000,25.000000,27.780000
25%,46.000000,94.500000,40.500000,41.085000
50%,49.000000,99.000000,50.000000,50.510000
75%,59.000000,99.000000,63.500000,62.970000
max,145.000000,249.000000,160.000000,80.400000


In [358]:
df_herb_seeds_copy.isna().sum()

Name_of_Seed                     0
No_of_Reviews                    5
Average_Rating                   0
Sale_Price                       0
Regular_Price                    0
Discount                         0
Discount_Percent                 0
Number of Seeds                  8
Seed Type                        9
Sowing Season                    8
Germination Temperature          8
Germination Time                 8
Germination Rate                 8
First Harvest                    9
Container Requirement / Pots    10
Growing Level                    8
Product_URL                      0
dtype: int64

In [359]:
df_micro_seeds_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Name_of_Seed             25 non-null     object 
 1   No_of_Reviews            13 non-null     object 
 2   Average_Rating           25 non-null     object 
 3   Sale_Price               25 non-null     float64
 4   Regular_Price            25 non-null     float64
 5   Discount                 25 non-null     float64
 6   Discount_Percent         25 non-null     float64
 7   Number of Seeds          19 non-null     object 
 8   Seed Type                19 non-null     object 
 9   Sowing Season            19 non-null     object 
 10  Germination Temperature  19 non-null     object 
 11  Germination Time         19 non-null     object 
 12  Germination Rate         19 non-null     object 
 13  First Harvest            19 non-null     object 
 14  Growing Level            19 

In [360]:
df_micro_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent
count,25.000000,25.000000,25.000000,25.000000
mean,103.720000,166.160000,62.440000,39.122000
std,62.477676,83.078116,31.136902,13.100272
min,39.000000,90.000000,1.000000,1.010000
25%,59.000000,99.000000,41.000000,33.440000
50%,79.000000,120.000000,59.000000,38.550000
75%,110.000000,199.000000,70.000000,50.170000
max,249.000000,349.000000,150.000000,56.670000


In [361]:
df_micro_seeds_copy.isna().sum()

Name_of_Seed                0
No_of_Reviews              12
Average_Rating              0
Sale_Price                  0
Regular_Price               0
Discount                    0
Discount_Percent            0
Number of Seeds             6
Seed Type                   6
Sowing Season               6
Germination Temperature     6
Germination Time            6
Germination Rate            6
First Harvest               6
Growing Level               6
Product_URL                 0
dtype: int64

In [362]:
df_organic_seeds_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  14 non-null     object 
 1   No_of_Reviews                 13 non-null     object 
 2   Average_Rating                14 non-null     object 
 3   Sale_Price                    14 non-null     float64
 4   Regular_Price                 14 non-null     float64
 5   Discount                      14 non-null     float64
 6   Discount_Percent              14 non-null     float64
 7   Number of Seeds               12 non-null     object 
 8   Seed Type                     12 non-null     object 
 9   Sowing Season                 12 non-null     object 
 10  Germination Temperature       12 non-null     object 
 11  Germination Time              12 non-null     object 
 12  Germination Rate              12 non-null     object 
 13  First H

In [363]:
df_organic_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent,Is_Organic
count,14.000000,14.000000,14.000000,14.000000,14.0
mean,65.500000,127.071429,61.571429,49.527857,1.0
std,38.878014,63.337271,28.966066,9.325384,0.0
min,30.000000,99.000000,40.000000,36.140000,1.0
25%,49.000000,99.000000,50.000000,44.417500,1.0
50%,49.000000,99.000000,50.000000,50.510000,1.0
75%,66.500000,110.000000,57.750000,50.510000,1.0
max,159.000000,299.000000,150.000000,72.730000,1.0


In [364]:
df_organic_seeds_copy.isna().sum()

Name_of_Seed                    0
No_of_Reviews                   1
Average_Rating                  0
Sale_Price                      0
Regular_Price                   0
Discount                        0
Discount_Percent                0
Number of Seeds                 2
Seed Type                       2
Sowing Season                   2
Germination Temperature         2
Germination Time                2
Germination Rate                2
First Harvest                   2
Container Requirement / Pots    4
Growing Level                   2
Product_URL                     0
Is_Organic                      0
dtype: int64

In [ ]:
df_winter_seeds_copy
df_winter_flower_seeds_copy
df_summer_vegetable_seeds_copy
df_summer_flower_seeds_copy
df_rainy_seeds_copy
df_other_seeds_copy

In [365]:
df_winter_seeds_copy.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  110 non-null    object 
 1   No_of_Reviews                 109 non-null    object 
 2   Average_Rating                110 non-null    object 
 3   Sale_Price                    110 non-null    float64
 4   Regular_Price                 110 non-null    float64
 5   Discount                      110 non-null    float64
 6   Discount_Percent              110 non-null    float64
 7   Number of Seeds               100 non-null    object 
 8   Seed Type                     101 non-null    object 
 9   Sowing Season                 102 non-null    object 
 10  Germination Temperature       102 non-null    object 
 11  Germination Time              102 non-null    object 
 12  Germination Rate              102 non-null    object 
 13  First

In [366]:
df_winter_seeds_copy.describe()

,Sale_Price,Regular_Price,Discount,Discount_Percent,Is_Winter
count,110.000000,110.000000,110.000000,110.000000,110.0
mean,73.181818,129.354545,56.172727,43.965455,1.0
std,58.858590,94.041528,42.224576,12.774257,0.0
min,30.000000,90.000000,20.000000,10.050000,1.0
25%,49.000000,99.000000,40.000000,34.340000,1.0
50%,59.000000,99.000000,50.000000,43.935000,1.0
75%,79.000000,149.000000,59.000000,50.510000,1.0
max,599.000000,999.000000,400.000000,75.380000,1.0


In [374]:
df_summer_vegetable_seeds_copy.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  157 non-null    object 
 1   No_of_Reviews                 154 non-null    object 
 2   Average_Rating                157 non-null    object 
 3   Sale_Price                    157 non-null    float64
 4   Regular_Price                 157 non-null    float64
 5   Discount                      157 non-null    float64
 6   Discount_Percent              157 non-null    float64
 7   Number of Seeds               132 non-null    object 
 8   Seed Type                     135 non-null    object 
 9   Sowing Season                 136 non-null    object 
 10  Germination Temperature       136 non-null    object 
 11  Germination Time              136 non-null    object 
 12  Germination Rate              136 non-null    object 
 13  First

In [375]:
df_summer_vegetable_seeds_copy.describe()


,Sale_Price,Regular_Price,Discount,Discount_Percent,Is_Summer
count,157.000000,157.000000,157.000000,157.000000,157.0
mean,77.713376,138.121019,60.407643,44.141401,1.0
std,66.019427,112.700553,53.532276,12.736193,0.0
min,30.000000,89.000000,20.000000,16.720000,1.0
25%,49.000000,99.000000,40.000000,34.340000,1.0
50%,59.000000,99.000000,50.000000,43.800000,1.0
75%,79.000000,149.000000,60.000000,50.750000,1.0
max,599.000000,999.000000,400.000000,75.380000,1.0


In [376]:

df_summer_vegetable_seeds_copy.isna().sum()

Name_of_Seed                      0
No_of_Reviews                     3
Average_Rating                    0
Sale_Price                        0
Regular_Price                     0
Discount                          0
Discount_Percent                  0
Number of Seeds                  25
Seed Type                        22
Sowing Season                    21
Germination Temperature          21
Germination Time                 21
Germination Rate                 21
First Harvest                    21
Container Requirement / Pots     22
Growing Level                    21
Water / Soaking Required        156
Product Ideal For               156
Product_URL                       0
Blooming Time                   156
Is_Summer                         0
dtype: int64

In [367]:
df_winter_seeds_copy.isna().sum()

Name_of_Seed                      0
No_of_Reviews                     1
Average_Rating                    0
Sale_Price                        0
Regular_Price                     0
Discount                          0
Discount_Percent                  0
Number of Seeds                  10
Seed Type                         9
Sowing Season                     8
Germination Temperature           8
Germination Time                  8
Germination Rate                  8
First Harvest                     8
Container Requirement / Pots      8
Growing Level                     8
Water / Soaking Required        109
Product Ideal For               109
Product_URL                       0
Is_Winter                         0
dtype: int64

In [368]:
df_winter_flower_seeds_copy.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  141 non-null    object 
 1   No_of_Reviews                 114 non-null    object 
 2   Average_Rating                141 non-null    object 
 3   Sale_Price                    141 non-null    float64
 4   Regular_Price                 141 non-null    float64
 5   Discount                      141 non-null    float64
 6   Discount_Percent              141 non-null    float64
 7   Number of Seeds               127 non-null    object 
 8   Seed Type                     133 non-null    object 
 9   Sowing Season                 133 non-null    object 
 10  Germination Temperature       133 non-null    object 
 11  Germination Time              133 non-null    object 
 12  Germination Rate              133 non-null    object 
 13  Bloom

In [369]:
df_winter_flower_seeds_copy.describe()


,Sale_Price,Regular_Price,Discount,Discount_Percent
count,141.000000,141.000000,141.000000,141.000000
mean,101.737589,173.361702,71.624113,43.696170
std,101.521683,159.714758,66.419397,12.048446
min,30.000000,70.000000,11.000000,12.220000
25%,49.000000,90.000000,40.000000,34.440000
50%,59.000000,99.000000,50.000000,40.400000
75%,99.000000,189.000000,69.000000,50.830000
max,499.000000,999.000000,500.000000,78.840000


In [370]:

df_winter_flower_seeds_copy.isna().sum()

Name_of_Seed                      0
No_of_Reviews                    27
Average_Rating                    0
Sale_Price                        0
Regular_Price                     0
Discount                          0
Discount_Percent                  0
Number of Seeds                  14
Seed Type                         8
Sowing Season                     8
Germination Temperature           8
Germination Time                  8
Germination Rate                  8
Blooming Time                    10
Container Requirement / Pots      9
Growing Level                     9
Product_URL                       0
Plant Size                      140
First Harvest                   138
dtype: int64

In [371]:
df_summer_flower_seeds_copy.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  70 non-null     object 
 1   No_of_Reviews                 64 non-null     object 
 2   Average_Rating                70 non-null     object 
 3   Sale_Price                    70 non-null     float64
 4   Regular_Price                 70 non-null     float64
 5   Discount                      70 non-null     float64
 6   Discount_Percent              70 non-null     float64
 7   Number of Seeds               56 non-null     object 
 8   Seed Type                     62 non-null     object 
 9   Sowing Season                 62 non-null     object 
 10  Germination Temperature       62 non-null     object 
 11  Germination Time              62 non-null     object 
 12  Germination Rate              62 non-null     object 
 13  Bloomin

In [372]:

df_summer_flower_seeds_copy.describe()


,Sale_Price,Regular_Price,Discount,Discount_Percent
count,70.000000,70.000000,70.000000,70.000000
mean,96.242857,167.985714,71.742857,41.316571
std,88.003530,167.013711,82.744882,10.677624
min,30.000000,70.000000,11.000000,12.220000
25%,59.000000,92.250000,35.000000,34.340000
50%,59.000000,99.000000,50.000000,40.400000
75%,95.500000,149.000000,58.500000,49.247500
max,499.000000,999.000000,500.000000,78.840000


In [373]:

df_summer_flower_seeds_copy.isna().sum()

Name_of_Seed                     0
No_of_Reviews                    6
Average_Rating                   0
Sale_Price                       0
Regular_Price                    0
Discount                         0
Discount_Percent                 0
Number of Seeds                 14
Seed Type                        8
Sowing Season                    8
Germination Temperature          8
Germination Time                 8
Germination Rate                 8
Blooming Time                   10
Container Requirement / Pots     9
Growing Level                    8
Product_URL                      0
First Harvest                   67
dtype: int64

In [377]:
df_rainy_seeds_copy.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  56 non-null     object 
 1   No_of_Reviews                 56 non-null     object 
 2   Average_Rating                56 non-null     object 
 3   Sale_Price                    56 non-null     float64
 4   Regular_Price                 56 non-null     float64
 5   Discount                      56 non-null     float64
 6   Discount_Percent              56 non-null     float64
 7   Number of Seeds               55 non-null     object 
 8   Seed Type                     56 non-null     object 
 9   Sowing Season                 56 non-null     object 
 10  Germination Temperature       56 non-null     object 
 11  Germination Time              56 non-null     object 
 12  Germination Rate              56 non-null     object 
 13  First H

In [378]:
df_rainy_seeds_copy.describe()


,Sale_Price,Regular_Price,Discount,Discount_Percent,Is_Rainy
count,56.000000,56.000000,56.000000,56.000000,56.0
mean,81.232143,130.178571,48.946429,37.997143,1.0
std,74.921047,121.789551,49.621996,10.268058,0.0
min,39.000000,90.000000,20.000000,17.500000,1.0
25%,55.000000,99.000000,31.000000,30.300000,1.0
50%,69.000000,99.000000,40.500000,35.175000,1.0
75%,80.500000,120.000000,50.000000,46.732500,1.0
max,599.000000,999.000000,400.000000,59.600000,1.0


In [379]:

df_rainy_seeds_copy.isna().sum()

Name_of_Seed                     0
No_of_Reviews                    0
Average_Rating                   0
Sale_Price                       0
Regular_Price                    0
Discount                         0
Discount_Percent                 0
Number of Seeds                  1
Seed Type                        0
Sowing Season                    0
Germination Temperature          0
Germination Time                 0
Germination Rate                 0
First Harvest                    0
Container Requirement / Pots     1
Growing Level                    0
Water / Soaking Required        55
Product Ideal For               55
Product_URL                      0
Is_Rainy                         0
dtype: int64

In [380]:
df_other_seeds_copy.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Name_of_Seed                  7 non-null      object 
 1   No_of_Reviews                 7 non-null      object 
 2   Average_Rating                7 non-null      object 
 3   Sale_Price                    7 non-null      float64
 4   Regular_Price                 7 non-null      float64
 5   Discount                      7 non-null      float64
 6   Discount_Percent              7 non-null      float64
 7   Number of Seeds               6 non-null      object 
 8   Seed Type                     6 non-null      object 
 9   Sowing Season                 6 non-null      object 
 10  Germination Temperature       6 non-null      object 
 11  Germination Time              6 non-null      object 
 12  Germination Rate              6 non-null      object 
 13  Container

In [381]:

df_other_seeds_copy.describe()


,Sale_Price,Regular_Price,Discount,Discount_Percent
count,7.000000,7.000000,7.000000,7.000000
mean,164.857143,280.428571,115.571429,47.834286
std,169.076342,197.520343,89.725242,23.273966
min,40.000000,99.000000,50.000000,16.690000
25%,49.000000,139.000000,50.000000,33.615000
50%,69.000000,189.000000,100.000000,50.510000
75%,224.000000,399.000000,129.500000,60.785000
max,499.000000,599.000000,300.000000,78.840000


In [382]:

df_other_seeds_copy.isna().sum()

Name_of_Seed                    0
No_of_Reviews                   0
Average_Rating                  0
Sale_Price                      0
Regular_Price                   0
Discount                        0
Discount_Percent                0
Number of Seeds                 1
Seed Type                       1
Sowing Season                   1
Germination Temperature         1
Germination Time                1
Germination Rate                1
Container Requirement / Pots    4
Growing Level                   1
Water / Soaking Required        6
Product_URL                     0
First Harvest                   4
dtype: int64

# AFTER CLEANING

In [14]:


df_all_seeds_copy = pd.read_csv("df_all_seeds_copy_uncleaned.csv")

df_best_seller_copy = pd.read_csv("df_best_seller_copy_uncleaned.csv")

df_bulbs_copy = pd.read_csv("df_bulbs_copy_uncleaned.csv")

df_exotic_seeds_copy = pd.read_csv("df_exotic_seeds_copy_uncleaned.csv")

df_fruit_seeds_copy = pd.read_csv("df_fruit_seeds_copy_uncleaned.csv")

df_herb_seeds_copy = pd.read_csv("df_herb_seeds_copy_uncleaned.csv")

df_micro_seeds_copy = pd.read_csv("df_micro_seeds_copy_uncleaned.csv")

df_organic_seeds_copy = pd.read_csv("df_organic_seeds_copy_uncleaned.csv")

df_other_seeds_copy = pd.read_csv("df_other_seeds_copy_uncleaned.csv")

df_rainy_flower_seeds_copy = pd.read_csv("df_rainy_flower_seeds_copy_uncleaned.csv")

df_summer_flower_seeds_copy = pd.read_csv("df_summer_flower_seeds_copy_uncleaned.csv")

df_winter_seeds_copy = pd.read_csv("df_winter_seeds_copy_uncleaned.csv")

df_winter_flower_seeds_copy = pd.read_csv("df_winter_flower_seeds_copy_uncleaned.csv")

df_summer_vegetable_seeds_copy = pd.read_csv("df_summer_vegetable_seeds_copy_uncleaned.csv")

In [33]:
df_rainy_seeds_copy = pd.read_csv("df_rainy_seeds_copy_uncleaned.csv")

In [17]:
cleaned_all_seeds = clean_scraped_dataframe(df_all_seeds_copy)

C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\3767414841.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(
C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\719497592.py:79: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doi

In [18]:
cleaned_all_seeds

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,Blooming Time
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,18–28°C,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/tomato-seed-...,45-60 Days
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/okra-or-lady...,45-60 Days
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",18-30 °C,6-12 Days,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,https://organicbazar.net/products/cucumber-see...,45-60 Days
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,18–28°C,7-21 Days,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/coriander-se...,45-60 Days
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/spinach-seeds,45-60 Days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,Salvia Vista Purple Flower Seeds (20 Seeds) Hi...,0,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Summer Season,",18–28°C,7-21 Days,Above 70%,60-80 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,60-80 Days
271,Knol Khol Purple Microgreen Seeds (25g),0,0.00,79.0,149.0,70.0,46.98,50 Seeds,"Open Pollinated,",All Season,15-26 °C,7–14 Days,Above 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/knol-khol-pu...,45-60 Days
272,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,0,0.00,95.0,199.0,104.0,52.26,30 Seeds,"Hybrid,","Winter Season,",15-26 °C,7–14 Days,Above 70%,60-80 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/matthiola-st...,60-80 Days
273,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,50 Seeds,"Open Pollinated,",All Season,15-26 °C,7–14 Days,Above 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/safed-musli-...,45-60 Days


In [19]:
cleaned_all_seeds = clean_scraped_dataframe(df_all_seeds_copy)

cleaned_best_seller = clean_scraped_dataframe(df_best_seller_copy)

cleaned_bulbs = clean_scraped_dataframe(df_bulbs_copy)

cleaned_exotic_seeds = clean_scraped_dataframe(df_exotic_seeds_copy)

cleaned_fruit_seeds = clean_scraped_dataframe(df_fruit_seeds_copy)

cleaned_herb_seeds = clean_scraped_dataframe(df_herb_seeds_copy)

cleaned_micro_seeds = clean_scraped_dataframe(df_micro_seeds_copy)

cleaned_organic_seeds = clean_scraped_dataframe(df_organic_seeds_copy)

cleaned_other_seeds = clean_scraped_dataframe(df_other_seeds_copy)

cleaned_rainy_flower_seeds = clean_scraped_dataframe(df_rainy_flower_seeds_copy)

cleaned_summer_flower_seeds = clean_scraped_dataframe(df_summer_flower_seeds_copy)

cleaned_winter_seeds = clean_scraped_dataframe(df_winter_seeds_copy)

cleaned_winter_flower_seeds = clean_scraped_dataframe(df_winter_flower_seeds_copy)

cleaned_summer_vegetable_seeds = clean_scraped_dataframe(df_summer_vegetable_seeds_copy)

C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\3767414841.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(
C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\719497592.py:79: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doi

In [34]:
cleaned_rainy_seeds = clean_scraped_dataframe(df_rainy_seeds_copy)

C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\3767414841.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(
C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\719497592.py:79: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doi

In [20]:
cleaned_all_seeds.isna().sum()

Name_of_Seed                    0
No_of_Reviews                   0
Average_Rating                  0
Sale_Price                      0
Regular_Price                   0
Discount                        0
Discount_Percent                0
Number of Seeds                 0
Seed Type                       0
Sowing Season                   0
Germination Temperature         0
Germination Time                0
Germination Rate                0
First Harvest                   0
Container Requirement / Pots    0
Growing Level                   0
Product_URL                     0
Blooming Time                   0
dtype: int64

In [21]:
cleaned_bulbs.isna().sum()

Name_of_Seed                    0
No_of_Reviews                   0
Average_Rating                  0
Sale_Price                      0
Regular_Price                   0
Discount                        0
Discount_Percent                0
Product_URL                     0
Number of Seeds                 0
Sowing Season                   0
Germination Temperature         0
Germination Time                0
Germination Rate                0
Blooming Time                   0
Container Requirement / Pots    0
Growing Level                   0
Seed Type                       0
dtype: int64

In [140]:
cleaned_all_seeds.to_csv('cleaned_all_seeds.csv', index = False)

cleaned_rainy_seeds.to_csv('cleaned_rainy_seeds.csv', index=False)

cleaned_best_seller.to_csv('cleaned_best_seller.csv', index=False)

cleaned_bulbs.to_csv('cleaned_bulbs.csv', index=False)

cleaned_exotic_seeds.to_csv('cleaned_exotic_seeds.csv', index=False)

cleaned_fruit_seeds.to_csv(
    'cleaned_fruit_seeds.csv',
    index=False
)

cleaned_herb_seeds.to_csv(
    'cleaned_herb_seeds.csv',
    index=False
)

cleaned_micro_seeds.to_csv(
    'cleaned_micro_seeds.csv',
    index=False
)

cleaned_organic_seeds.to_csv(
    'cleaned_organic_seeds.csv',
    index=False
)

cleaned_other_seeds.to_csv(
    'cleaned_other_seeds.csv',
    index=False
)

cleaned_rainy_flower_seeds.to_csv(
    'cleaned_rainy_flower_seeds.csv',
    index=False
)

cleaned_summer_flower_seeds.to_csv(
    'cleaned_summer_flower_seeds.csv',
    index=False
)

cleaned_winter_seeds.to_csv(
    'cleaned_winter_seeds.csv',
    index=False
)

cleaned_winter_flower_seeds.to_csv(
    'cleaned_winter_flower_seeds.csv',
    index=False
)

cleaned_summer_vegetable_seeds.to_csv(
    'cleaned_summer_vegetable_seeds.csv',
    index=False
)

In [79]:
cleaned_all_seeds['Is_All'] = 1

cleaned_best_seller['Is_Best_Seller'] = 1

cleaned_bulbs['Is_Bulbs'] = 1

cleaned_exotic_seeds['Is_Exotic_Seeds'] = 1

cleaned_fruit_seeds['Is_Fruit_Seeds'] = 1

cleaned_herb_seeds['Is_Herb_Seeds'] = 1

cleaned_micro_seeds['Is_Micro_Seeds'] = 1

cleaned_organic_seeds['Is_Organic_Seeds'] = 1

cleaned_other_seeds['Is_Other_Seeds'] = 1

cleaned_rainy_seeds['Is_Rainy_Seeds'] = 1

cleaned_rainy_flower_seeds['Is_Rainy_Flower_Seeds'] = 1

cleaned_summer_flower_seeds['Is_Summer_Flower_Seeds'] = 1

cleaned_winter_flower_seeds['Is_Winter_Flower_Seeds'] = 1

cleaned_summer_vegetable_seeds['Is_Summer_Vegetable_Seeds'] = 1

cleaned_winter_seeds['Is_Winter_Seeds'] = 1

In [161]:
cleaned_winter_flower_seeds

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Product_URL,Is_Winter_Flower_Seeds
0,Daisy Pomponette Mix Seeds - 300 Seeds (Gulbah...,58.0,4.66,45.0,149.0,104.0,69.80,300 Seeds,"Open Pollinated,","Winter Season,",15–20°C,10 - 21 Days,Over 70%,60-90 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/daisy-pompon...,1
1,Marigold Orange Flower Seeds - 100 Seeds (Gend...,43.0,4.67,65.0,99.0,34.0,34.34,100 Seeds,"Open Pollinated,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"24x9 inch, 18x6 inch, 24x6 inch, 24x12 inch",Easy – Ideal for beginners,https://organicbazar.net/products/marigold-see...,1
2,Zinnia Mix Color Flower Seeds (100 Seeds) High...,59.0,4.63,76.0,110.0,34.0,30.91,100 Seeds,"Open Pollinated,",All Season,18-25°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/zinnia-doubl...,1
3,Stock Mixed Color Flower Seeds (100 Seeds) Hig...,44.0,4.73,69.0,99.0,30.0,30.30,100 Seeds,"Open Pollinated,","Winter Season,",18-25°C,7–14 Days,Above 70%,40-60 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/stocks-doubl...,1
4,Marigold White Flower Seeds - 50 Seeds (Genda/...,48.0,4.54,88.0,120.0,32.0,26.67,50 Seeds,"Open Pollinated,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/marigold-dou...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,Torenia Kauai Mix Color Flower Seeds (30 Seeds...,15.0,0.00,199.0,299.0,100.0,33.44,30 Seeds,"Open Pollinated,","Rainy Season,",18-30 °C,7–14 Days,Above 70%,60-80 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/torenia-kaua...,1
137,Salvia Vista Red Flower Seeds (20 Seeds) High ...,15.0,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Winter Season,",18–28°C,7-21 Days,Above 70%,60-80 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,1
138,Coleus Wizard Mix Seeds (100 Seeds) High Germi...,15.0,0.00,185.0,299.0,114.0,38.13,100 Seeds,"Open Pollinated,","Rainy Season,",18–28°C,7–14 Days,Above 70%,60-120 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/coleus-wizar...,1
139,Celosia Ice Cream Yellow Flower Seeds (50 Seed...,15.0,0.00,143.0,269.0,126.0,46.84,50 Seeds,"Hybrid,","Summer Season,",18-30 °C,7–14 Days,Above 70%,60-80 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/celosia-ice-...,1


In [80]:
df_combine_clean =  pd.concat([

    cleaned_all_seeds,

    cleaned_best_seller,

    cleaned_bulbs,

    cleaned_exotic_seeds,

    cleaned_fruit_seeds,

    cleaned_herb_seeds,

    cleaned_micro_seeds,

    cleaned_organic_seeds,

    cleaned_other_seeds,

    cleaned_rainy_seeds,

    cleaned_rainy_flower_seeds,

    cleaned_summer_flower_seeds,

    cleaned_winter_flower_seeds,

    cleaned_summer_vegetable_seeds,

    cleaned_winter_seeds

], ignore_index=True)


In [81]:
df_combine_clean.isna().sum()

Name_of_Seed                       0
No_of_Reviews                      0
Average_Rating                     0
Sale_Price                         0
Regular_Price                      0
Discount                           0
Discount_Percent                   0
Number of Seeds                    0
Seed Type                          0
Sowing Season                      0
Germination Temperature            0
Germination Time                   0
Germination Rate                   0
First Harvest                    370
Container Requirement / Pots      25
Growing Level                      0
Product_URL                        0
Blooming Time                    431
Is_All                           857
Product Ideal For               1076
Material                        1076
GSM                             1076
Vegetables                      1076
Flowers                         1076
Drainage                        1076
Capacity                        1076
Durability                      1076
H

In [82]:
df_combine_clean = df_combine_clean.drop_duplicates()

In [83]:
df_combine_clean['Is_Exotic'] = df_combine_clean['Is_Exotic'].fillna(0)
df_combine_clean['Is_Organic'] = df_combine_clean['Is_Organic'].fillna(0)
df_combine_clean['Is_Summer'] = df_combine_clean['Is_Summer'].fillna(0)
df_combine_clean['Is_Winter'] = df_combine_clean['Is_Winter'].fillna(0)
df_combine_clean['Is_All'] = df_combine_clean['Is_All'].fillna(0)

df_combine_clean['Is_Best_Seller'] = df_combine_clean['Is_Best_Seller'].fillna(0)

df_combine_clean['Is_Bulbs'] = df_combine_clean['Is_Bulbs'].fillna(0)

df_combine_clean['Is_Exotic_Seeds'] = df_combine_clean['Is_Exotic_Seeds'].fillna(0)

df_combine_clean['Is_Fruit_Seeds'] = df_combine_clean['Is_Fruit_Seeds'].fillna(0)

df_combine_clean['Is_Herb_Seeds'] = df_combine_clean['Is_Herb_Seeds'].fillna(0)

df_combine_clean['Is_Micro_Seeds'] = df_combine_clean['Is_Micro_Seeds'].fillna(0)

df_combine_clean['Is_Organic_Seeds'] = df_combine_clean['Is_Organic_Seeds'].fillna(0)

df_combine_clean['Is_Other_Seeds'] = df_combine_clean['Is_Other_Seeds'].fillna(0)

df_combine_clean['Is_Rainy_Seeds'] = df_combine_clean['Is_Rainy_Seeds'].fillna(0)

df_combine_clean['Is_Rainy_Flower_Seeds'] = df_combine_clean['Is_Rainy_Flower_Seeds'].fillna(0)

df_combine_clean['Is_Summer_Flower_Seeds'] = df_combine_clean['Is_Summer_Flower_Seeds'].fillna(0)

df_combine_clean['Is_Winter_Flower_Seeds'] = df_combine_clean['Is_Winter_Flower_Seeds'].fillna(0)

df_combine_clean['Is_Summer_Vegetable_Seeds'] = df_combine_clean['Is_Summer_Vegetable_Seeds'].fillna(0)

df_combine_clean['Is_Winter_Seeds'] = df_combine_clean['Is_Winter_Seeds'].fillna(0)


In [84]:
df_combine_clean.isna().sum()

Name_of_Seed                       0
No_of_Reviews                      0
Average_Rating                     0
Sale_Price                         0
Regular_Price                      0
Discount                           0
Discount_Percent                   0
Number of Seeds                    0
Seed Type                          0
Sowing Season                      0
Germination Temperature            0
Germination Time                   0
Germination Rate                   0
First Harvest                    370
Container Requirement / Pots      25
Growing Level                      0
Product_URL                        0
Blooming Time                    431
Is_All                             0
Product Ideal For               1076
Material                        1076
GSM                             1076
Vegetables                      1076
Flowers                         1076
Drainage                        1076
Capacity                        1076
Durability                      1076
H

In [85]:
len(df_combine_clean)

1132

In [86]:
df_combine_clean.columns

Index(['Name_of_Seed', 'No_of_Reviews', 'Average_Rating', 'Sale_Price',
       'Regular_Price', 'Discount', 'Discount_Percent', 'Number of Seeds',
       'Seed Type', 'Sowing Season', 'Germination Temperature',
       'Germination Time', 'Germination Rate', 'First Harvest',
       'Container Requirement / Pots', 'Growing Level', 'Product_URL',
       'Blooming Time', 'Is_All', 'Product Ideal For', 'Material', 'GSM',
       'Vegetables', 'Flowers', 'Drainage', 'Capacity', 'Durability',
       'Handles', 'Special Benefits', 'Shape', 'Color', 'Accessories Type',
       'Is_Best_Seller', 'Is_Bulbs', 'Is_Exotic', 'Is_Exotic_Seeds',
       'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds', 'Is_Organic',
       'Is_Organic_Seeds', 'Water / Soaking Required', 'Is_Other_Seeds',
       'Is_Rainy', 'Is_Rainy_Seeds', 'Is_Rainy_Flower_Seeds',
       'Is_Summer_Flower_Seeds', 'Is_Winter_Flower_Seeds', 'Is_Summer',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter', 'Is_Winter_Seeds'],
      dtype='ob

In [87]:
df_combine_clean['Is_Summer'].value_counts()

Is_Summer
0.0    975
1.0    157
Name: count, dtype: int64

In [88]:
cleaned_all_seeds

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,First Harvest,Container Requirement / Pots,Growing Level,Product_URL,Blooming Time,Is_All
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,18–28°C,10- 15 Days,Over 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/tomato-seed-...,45-60 Days,1
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",25-35°C,7–14 Days,Above 70%,45-70 Days,"21x21 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/okra-or-lady...,45-60 Days,1
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",18-30 °C,6-12 Days,Above 70%,50-60 Days,24x24 inch,Easy – Ideal for beginners,https://organicbazar.net/products/cucumber-see...,45-60 Days,1
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,18–28°C,7-21 Days,Above 70%,40-45 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/coriander-se...,45-60 Days,1
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,18–28°C,7–14 Days,Over 70%,30–45 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/spinach-seeds,45-60 Days,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,Salvia Vista Purple Flower Seeds (20 Seeds) Hi...,0,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Summer Season,",18–28°C,7-21 Days,Above 70%,60-80 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,60-80 Days,1
271,Knol Khol Purple Microgreen Seeds (25g),0,0.00,79.0,149.0,70.0,46.98,50 Seeds,"Open Pollinated,",All Season,15-26 °C,7–14 Days,Above 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/knol-khol-pu...,45-60 Days,1
272,Matthiola (Stock) Katz Pink Cut Flower Seeds (...,0,0.00,95.0,199.0,104.0,52.26,30 Seeds,"Hybrid,","Winter Season,",15-26 °C,7–14 Days,Above 70%,60-80 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/matthiola-st...,60-80 Days,1
273,Safed Musli Bulbs Seeds,1,5.00,99.0,149.0,50.0,33.56,50 Seeds,"Open Pollinated,",All Season,15-26 °C,7–14 Days,Above 70%,60-80 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/safed-musli-...,45-60 Days,1


In [89]:
cleaned_winter_flower_seeds

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Product_URL,Is_Winter_Flower_Seeds
0,Daisy Pomponette Mix Seeds - 300 Seeds (Gulbah...,58.0,4.66,45.0,149.0,104.0,69.80,300 Seeds,"Open Pollinated,","Winter Season,",15–20°C,10 - 21 Days,Over 70%,60-90 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/daisy-pompon...,1
1,Marigold Orange Flower Seeds - 100 Seeds (Gend...,43.0,4.67,65.0,99.0,34.0,34.34,100 Seeds,"Open Pollinated,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"24x9 inch, 18x6 inch, 24x6 inch, 24x12 inch",Easy – Ideal for beginners,https://organicbazar.net/products/marigold-see...,1
2,Zinnia Mix Color Flower Seeds (100 Seeds) High...,59.0,4.63,76.0,110.0,34.0,30.91,100 Seeds,"Open Pollinated,",All Season,18-25°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/zinnia-doubl...,1
3,Stock Mixed Color Flower Seeds (100 Seeds) Hig...,44.0,4.73,69.0,99.0,30.0,30.30,100 Seeds,"Open Pollinated,","Winter Season,",18-25°C,7–14 Days,Above 70%,40-60 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/stocks-doubl...,1
4,Marigold White Flower Seeds - 50 Seeds (Genda/...,48.0,4.54,88.0,120.0,32.0,26.67,50 Seeds,"Open Pollinated,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/marigold-dou...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,Torenia Kauai Mix Color Flower Seeds (30 Seeds...,15.0,0.00,199.0,299.0,100.0,33.44,30 Seeds,"Open Pollinated,","Rainy Season,",18-30 °C,7–14 Days,Above 70%,60-80 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/torenia-kaua...,1
137,Salvia Vista Red Flower Seeds (20 Seeds) High ...,15.0,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Winter Season,",18–28°C,7-21 Days,Above 70%,60-80 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,1
138,Coleus Wizard Mix Seeds (100 Seeds) High Germi...,15.0,0.00,185.0,299.0,114.0,38.13,100 Seeds,"Open Pollinated,","Rainy Season,",18–28°C,7–14 Days,Above 70%,60-120 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/coleus-wizar...,1
139,Celosia Ice Cream Yellow Flower Seeds (50 Seed...,15.0,0.00,143.0,269.0,126.0,46.84,50 Seeds,"Hybrid,","Summer Season,",18-30 °C,7–14 Days,Above 70%,60-80 Days,"12x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/celosia-ice-...,1


# Feature Selection

In [90]:
df_combine_clean

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,...,Is_Other_Seeds,Is_Rainy,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Winter_Flower_Seeds,Is_Summer,Is_Summer_Vegetable_Seeds,Is_Winter,Is_Winter_Seeds
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,103.0,4.11,69.0,99.0,30.0,30.30,50 Seeds,F-1 Hybrid,All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,73.0,4.30,65.0,99.0,34.0,34.34,50 Seeds,"Hybrid,","Rainy Season,",...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,44.0,4.27,69.0,99.0,30.0,30.30,20 Seeds,"Hybrid,","Rainy Season,",...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,88.0,4.77,59.0,99.0,40.0,40.40,1000 Seeds,"Open Pollinated,",All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,81.0,4.72,65.0,99.0,34.0,34.34,750 Seeds,"Open Pollinated,",All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green (Thick wall fruit...,2.0,1.00,79.0,149.0,70.0,46.98,50 Seeds,"Open Pollinated,",All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1128,Bitter Gourd (Barahmasi) Seeds - 12 Seeds (kar...,5.0,4.20,59.0,99.0,40.0,40.40,50 Seeds,"Open Pollinated,",All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1129,Salad Cabbage F1 Hybrid Seeds - 100 Seeds (Pat...,2.0,4.00,69.0,149.0,80.0,53.69,50 Seeds,"Open Pollinated,",All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1130,Collard Seeds (300 Seeds),3.0,5.00,49.0,99.0,50.0,50.51,50 Seeds,"Open Pollinated,",All Season,...,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [91]:
feature_selected_df = df_combine_clean[[
    
    # =====================================================
    # PRODUCT IDENTIFICATION
    # =====================================================
    
    "Name_of_Seed",
    
    "Seed Type",
    
    "Number of Seeds",
    
    "Sowing Season",
    
    
    # =====================================================
    # CUSTOMER ENGAGEMENT
    # =====================================================
    
    "No_of_Reviews",
    
    "Average_Rating",
    
    
    # =====================================================
    # PRICING FEATURES
    # =====================================================
    
    "Sale_Price",
    
    "Regular_Price",
    
    "Discount",
    
    "Discount_Percent",
    
    
    # =====================================================
    # CATEGORY FLAGS / BOOLEAN FEATURES
    # =====================================================
    
    "Is_All",
    
    "Is_Best_Seller",
    
    "Is_Bulbs",
    
    "Is_Exotic_Seeds",
    
    "Is_Fruit_Seeds",
    
    "Is_Herb_Seeds",
    
    "Is_Micro_Seeds",
    
    "Is_Organic_Seeds",
    
    "Is_Other_Seeds",
    
    "Is_Rainy_Seeds",
    
    "Is_Rainy_Flower_Seeds",
    
    "Is_Summer_Flower_Seeds",
    
    "Is_Summer_Vegetable_Seeds",
    
    "Is_Winter_Flower_Seeds",
    
    "Is_Winter_Seeds"

]]

In [92]:
feature_selected_df.reset_index(drop=True, inplace=True)

In [93]:
feature_selected_df

,Name_of_Seed,Seed Type,Number of Seeds,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,...,Is_Herb_Seeds,Is_Micro_Seeds,Is_Organic_Seeds,Is_Other_Seeds,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,F-1 Hybrid,50 Seeds,All Season,103.0,4.11,69.0,99.0,30.0,30.30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,"Hybrid,",50 Seeds,"Rainy Season,",73.0,4.30,65.0,99.0,34.0,34.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,"Hybrid,",20 Seeds,"Rainy Season,",44.0,4.27,69.0,99.0,30.0,30.30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,"Open Pollinated,",1000 Seeds,All Season,88.0,4.77,59.0,99.0,40.0,40.40,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,"Open Pollinated,",750 Seeds,All Season,81.0,4.72,65.0,99.0,34.0,34.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green (Thick wall fruit...,"Open Pollinated,",50 Seeds,All Season,2.0,1.00,79.0,149.0,70.0,46.98,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1128,Bitter Gourd (Barahmasi) Seeds - 12 Seeds (kar...,"Open Pollinated,",50 Seeds,All Season,5.0,4.20,59.0,99.0,40.0,40.40,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1129,Salad Cabbage F1 Hybrid Seeds - 100 Seeds (Pat...,"Open Pollinated,",50 Seeds,All Season,2.0,4.00,69.0,149.0,80.0,53.69,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1130,Collard Seeds (300 Seeds),"Open Pollinated,",50 Seeds,All Season,3.0,5.00,49.0,99.0,50.0,50.51,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [94]:
feature_selected_df.columns

Index(['Name_of_Seed', 'Seed Type', 'Number of Seeds', 'Sowing Season',
       'No_of_Reviews', 'Average_Rating', 'Sale_Price', 'Regular_Price',
       'Discount', 'Discount_Percent', 'Is_All', 'Is_Best_Seller', 'Is_Bulbs',
       'Is_Exotic_Seeds', 'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds',
       'Is_Organic_Seeds', 'Is_Other_Seeds', 'Is_Rainy_Seeds',
       'Is_Rainy_Flower_Seeds', 'Is_Summer_Flower_Seeds',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter_Flower_Seeds',
       'Is_Winter_Seeds'],
      dtype='object')

In [95]:
feature_selected_df['Name_of_Seed'].value_counts()

Name_of_Seed
Pumpkin (kaddu) Seeds - 12 Seeds (कद्दू के बीज) Easy To grow, High Germination, High Yield Kaddu Seeds for Home Gardening                       6
Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (टमाटर के बीज) High Germination/ Easy to Grow/ Ideal for Terrace/Kitchen Gardening                  5
Zinnia Mix Color Flower Seeds (100 Seeds) High Germination/ Easy To grow/ Perfect for pots, balconies, or Terrace gardens                       5
Marigold Mixed Color Seeds - 100 Seeds (Genda/ गेंदा के बीज) High Germination/ Easy To grow/ Perfect for pots, balconies, or Terrace gardens    5
Chilli F-1 Hybrid Seeds (Mirch) Seeds - (50 Seeds) High Yield | High Germination | Best for Home Gardens (मिर्च के बीज)                         5
                                                                                                                                               ..
Allium Molly (Yellow) Flower Bulbs (05 Bulbs)                                                                  

In [97]:
feature_selected_df['Clean_Name'] = feature_selected_df['Name_of_Seed']

# remove hindi
feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
    r'[\u0900-\u097F]+',
    '',
    regex=True
)

# remove emojis/special chars
feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
    r'[^\w\s\-\(\)]',
    '',
    regex=True
)

# remove marketing phrases
patterns = [
    r'High Germination.*',
    r'Easy To grow.*',
    r'Easy to Grow.*',
    r'High Yield.*',
    r'Ideal for.*',
    r'Perfect for.*',
    r'Best for.*'
]

for p in patterns:

    feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
        p,
        '',
        regex=True,
        case=False
    )

# remove seed counts
feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
    r'\(?\d+\s*Seeds?\)?',
    '',
    regex=True,
    case=False
)

# remove brackets
feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
    r'[\(\)]',
    '',
    regex=True
)

# clean spaces
feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
    r'\s+',
    ' ',
    regex=True
).str.strip()

C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\1233201897.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_selected_df['Clean_Name'] = feature_selected_df['Name_of_Seed']
C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\1233201897.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_selected_df['Clean_Name'] = feature_selected_df['Clean_Name'].str.replace(
C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\1233201897.py:11: SettingWithCopyWarning: 
A value is 

In [98]:
feature_selected_df

,Name_of_Seed,Seed Type,Number of Seeds,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,...,Is_Micro_Seeds,Is_Organic_Seeds,Is_Other_Seeds,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Clean_Name
0,Tomato F1 Hybrid Seeds - 50 Seeds (Tamatar) (ट...,F-1 Hybrid,50 Seeds,All Season,103.0,4.11,69.0,99.0,30.0,30.30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Tomato F1 Hybrid Seeds - Tamatar
1,Okra or Lady Finger Hybrid (bhindi) Seeds - 50...,"Hybrid,",50 Seeds,"Rainy Season,",73.0,4.30,65.0,99.0,34.0,34.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Okra or Lady Finger Hybrid bhindi Seeds -
2,Cucumber (Kheera) Hybrid Seeds - 20 Seeds (Kak...,"Hybrid,",20 Seeds,"Rainy Season,",44.0,4.27,69.0,99.0,30.0,30.30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cucumber Kheera Hybrid Seeds - Kakdi -
3,Coriander (Dhaniya) Seeds - 1000 Seeds (धनिया ...,"Open Pollinated,",1000 Seeds,All Season,88.0,4.77,59.0,99.0,40.0,40.40,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Coriander Dhaniya Seeds -
4,Spinach (Palak) Seeds (750 Seeds) (पालक के बीज...,"Open Pollinated,",750 Seeds,All Season,81.0,4.72,65.0,99.0,34.0,34.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Spinach Palak Seeds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green (Thick wall fruit...,"Open Pollinated,",50 Seeds,All Season,2.0,1.00,79.0,149.0,70.0,46.98,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Chilli Yellowish Light Green Thick wall fruits...
1128,Bitter Gourd (Barahmasi) Seeds - 12 Seeds (kar...,"Open Pollinated,",50 Seeds,All Season,5.0,4.20,59.0,99.0,40.0,40.40,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Bitter Gourd Barahmasi Seeds - karela
1129,Salad Cabbage F1 Hybrid Seeds - 100 Seeds (Pat...,"Open Pollinated,",50 Seeds,All Season,2.0,4.00,69.0,149.0,80.0,53.69,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi
1130,Collard Seeds (300 Seeds),"Open Pollinated,",50 Seeds,All Season,3.0,5.00,49.0,99.0,50.0,50.51,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Collard Seeds


In [99]:
feature_selected_df['Sowing Season'].value_counts()

Sowing Season
All Season                         584
Winter Season,                     359
Rainy Season,                       97
Summer Season,                      89
Winter Season And Summer Season      2
Summer Season and Rainy Season       1
Name: count, dtype: int64

In [100]:
feature_selected_df['Sowing Season'] = (
    
    feature_selected_df['Sowing Season']
    
    .str.replace(',', '', regex=False)
    
    .str.strip()
)

C:\Users\Venkata Srikar\AppData\Local\Temp\ipykernel_16120\142118169.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_selected_df['Sowing Season'] = (


## Creating Season Bools based on Sowing Season name

In [164]:
feature_selected_df['Is_Summer'] = (
    
    feature_selected_df['Sowing Season']
    
    .str.contains('Summer', case=False, na=False)
    
    .astype(int)
)

feature_selected_df['Is_Winter'] = (
    
    feature_selected_df['Sowing Season']
    
    .str.contains('Winter', case=False, na=False)
    
    .astype(int)
)

feature_selected_df['Is_Rainy'] = (
    
    feature_selected_df['Sowing Season']
    
    .str.contains('Rainy', case=False, na=False)
    
    .astype(int)
)

all_season_mask = (
    
    feature_selected_df['Sowing Season']
    
    .str.contains('All Season', case=False, na=False)
)

feature_selected_df.loc[
    
    all_season_mask,
    
    ['Is_Summer', 'Is_Winter', 'Is_Rainy']
    
] = 1



In [113]:
feature_selected_df['Number of Seeds'] = (
    
    feature_selected_df['Number of Seeds']
    
    .str.extract(r'(\d+)')[0]
    
    .astype(float)
)

AttributeError: Can only use .str accessor with string values!

In [114]:
feature_selected_df.columns

Index(['Name_of_Seed', 'Seed Type', 'Number of Seeds', 'Sowing Season',
       'No_of_Reviews', 'Average_Rating', 'Sale_Price', 'Regular_Price',
       'Discount', 'Discount_Percent', 'Is_All', 'Is_Best_Seller', 'Is_Bulbs',
       'Is_Exotic_Seeds', 'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds',
       'Is_Organic_Seeds', 'Is_Other_Seeds', 'Is_Rainy_Seeds',
       'Is_Rainy_Flower_Seeds', 'Is_Summer_Flower_Seeds',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter_Flower_Seeds',
       'Is_Winter_Seeds', 'Clean_Name', 'Is_Summer', 'Is_Winter', 'Is_Rainy'],
      dtype='object')

In [115]:
feature_selected_df = feature_selected_df[['Clean_Name','Number of Seeds', 'Seed Type', 'Sowing Season',
                                           'No_of_Reviews', 'Average_Rating', 'Sale_Price', 'Regular_Price',
       'Discount', 'Discount_Percent',
        'Is_All', 'Is_Best_Seller', 'Is_Bulbs',
       'Is_Exotic_Seeds', 'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds',
       'Is_Organic_Seeds', 'Is_Other_Seeds', 'Is_Rainy_Seeds',
       'Is_Rainy_Flower_Seeds', 'Is_Summer_Flower_Seeds',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter_Flower_Seeds',
       'Is_Winter_Seeds',  'Is_Summer', 'Is_Winter', 'Is_Rainy']]

## Sale Price per 100 Seeds

In [117]:
feature_selected_df['Sale_Price_per_100_Seeds'] = (
    
    feature_selected_df['Sale_Price']/ feature_selected_df['Number of Seeds']
    
) * 100

In [123]:
feature_selected_df["Sale_Price_per_100_Seeds"] = np.round(feature_selected_df["Sale_Price_per_100_Seeds"],2) 

# ADD PER 100 SEEDS COLUMN

In [125]:
feature_selected_df['Sale_Price_per_100_Seeds']

0       138.00
1       130.00
2       345.00
3         5.90
4         8.67
         ...  
1127    158.00
1128    118.00
1129    138.00
1130     98.00
1131     59.00
Name: Sale_Price_per_100_Seeds, Length: 1132, dtype: float64

In [133]:
feature_selected_df.loc[:,'Clean_Name':'Sale_Price_per_100_Seeds']

,Clean_Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Sale_Price_per_100_Seeds
0,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,99.0,30.0,30.30,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,138.00
1,Okra or Lady Finger Hybrid bhindi Seeds -,50.0,"Hybrid,",Rainy Season,73.0,4.30,65.0,99.0,34.0,34.34,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,130.00
2,Cucumber Kheera Hybrid Seeds - Kakdi -,20.0,"Hybrid,",Rainy Season,44.0,4.27,69.0,99.0,30.0,30.30,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,345.00
3,Coriander Dhaniya Seeds -,1000.0,"Open Pollinated,",All Season,88.0,4.77,59.0,99.0,40.0,40.40,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,5.90
4,Spinach Palak Seeds,750.0,"Open Pollinated,",All Season,81.0,4.72,65.0,99.0,34.0,34.34,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,8.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green Thick wall fruits...,50.0,"Open Pollinated,",All Season,2.0,1.00,79.0,149.0,70.0,46.98,...,0.0,0.0,0.0,0.0,0.0,1.0,1,1,1,158.00
1128,Bitter Gourd Barahmasi Seeds - karela,50.0,"Open Pollinated,",All Season,5.0,4.20,59.0,99.0,40.0,40.40,...,0.0,0.0,0.0,0.0,0.0,1.0,1,1,1,118.00
1129,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi,50.0,"Open Pollinated,",All Season,2.0,4.00,69.0,149.0,80.0,53.69,...,0.0,0.0,0.0,0.0,0.0,1.0,1,1,1,138.00
1130,Collard Seeds,50.0,"Open Pollinated,",All Season,3.0,5.00,49.0,99.0,50.0,50.51,...,0.0,0.0,0.0,0.0,0.0,1.0,1,1,1,98.00


In [134]:
feature_selected_df = feature_selected_df[['Clean_Name','Number of Seeds', 'Seed Type', 'Sowing Season',
                                           'No_of_Reviews', 'Average_Rating', 'Sale_Price','Sale_Price_per_100_Seeds', 'Regular_Price',
       'Discount', 'Discount_Percent',
        'Is_All', 'Is_Best_Seller', 'Is_Bulbs',
       'Is_Exotic_Seeds', 'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds',
       'Is_Organic_Seeds', 'Is_Other_Seeds', 'Is_Rainy_Seeds',
       'Is_Rainy_Flower_Seeds', 'Is_Summer_Flower_Seeds',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter_Flower_Seeds',
       'Is_Winter_Seeds',  'Is_Summer', 'Is_Winter', 'Is_Rainy']]

In [136]:
feature_selected_df.loc[:,'Clean_Name':'Discount_Percent']

,Clean_Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,Discount_Percent
0,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,138.00,99.0,30.0,30.30
1,Okra or Lady Finger Hybrid bhindi Seeds -,50.0,"Hybrid,",Rainy Season,73.0,4.30,65.0,130.00,99.0,34.0,34.34
2,Cucumber Kheera Hybrid Seeds - Kakdi -,20.0,"Hybrid,",Rainy Season,44.0,4.27,69.0,345.00,99.0,30.0,30.30
3,Coriander Dhaniya Seeds -,1000.0,"Open Pollinated,",All Season,88.0,4.77,59.0,5.90,99.0,40.0,40.40
4,Spinach Palak Seeds,750.0,"Open Pollinated,",All Season,81.0,4.72,65.0,8.67,99.0,34.0,34.34
...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green Thick wall fruits...,50.0,"Open Pollinated,",All Season,2.0,1.00,79.0,158.00,149.0,70.0,46.98
1128,Bitter Gourd Barahmasi Seeds - karela,50.0,"Open Pollinated,",All Season,5.0,4.20,59.0,118.00,99.0,40.0,40.40
1129,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi,50.0,"Open Pollinated,",All Season,2.0,4.00,69.0,138.00,149.0,80.0,53.69
1130,Collard Seeds,50.0,"Open Pollinated,",All Season,3.0,5.00,49.0,98.00,99.0,50.0,50.51


In [165]:
feature_selected_df

,Seed Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Regular_Price_per_100_Seeds
0,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,138.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1
1,Okra or Lady Finger Hybrid bhindi Seeds -,50.0,Hybrid,Rainy Season,73.0,4.30,65.0,130.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1
2,Cucumber Kheera Hybrid Seeds - Kakdi -,20.0,Hybrid,Rainy Season,44.0,4.27,69.0,345.00,99.0,495.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,1
3,Coriander Dhaniya Seeds -,1000.0,Open Pollinated,All Season,88.0,4.77,59.0,5.90,99.0,9.9,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1
4,Spinach Palak Seeds,750.0,Open Pollinated,All Season,81.0,4.72,65.0,8.67,99.0,13.2,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green Thick wall fruits...,50.0,Open Pollinated,All Season,2.0,1.00,79.0,158.00,149.0,298.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1
1128,Bitter Gourd Barahmasi Seeds - karela,50.0,Open Pollinated,All Season,5.0,4.20,59.0,118.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1
1129,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi,50.0,Open Pollinated,All Season,2.0,4.00,69.0,138.00,149.0,298.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1
1130,Collard Seeds,50.0,Open Pollinated,All Season,3.0,5.00,49.0,98.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1


In [206]:
feature_selected_df.to_csv('CLEANED_Feature_Selected_Df.csv')

In [141]:
feature_selected_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1132 entries, 0 to 1131
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Clean_Name                 1132 non-null   object 
 1   Number of Seeds            1132 non-null   float64
 2   Seed Type                  1132 non-null   object 
 3   Sowing Season              1132 non-null   object 
 4   No_of_Reviews              1132 non-null   float64
 5   Average_Rating             1132 non-null   float64
 6   Sale_Price                 1132 non-null   float64
 7   Sale_Price_per_100_Seeds   1132 non-null   float64
 8   Regular_Price              1132 non-null   float64
 9   Discount                   1132 non-null   float64
 10  Discount_Percent           1132 non-null   float64
 11  Is_All                     1132 non-null   float64
 12  Is_Best_Seller             1132 non-null   float64
 13  Is_Bulbs                   1132 non-null   float

In [148]:
feature_selected_df = feature_selected_df.rename(columns={'Clean_Name': 'Seed Name'})

In [204]:
feature_selected_df[feature_selected_df['Is_Bulbs'] == 0]

,Seed Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Regular_Price_per_100_Seeds
0,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,138.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.0
1,Okra or Lady Finger Hybrid bhindi Seeds -,50.0,Hybrid,Rainy Season,73.0,4.30,65.0,130.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,198.0
2,Cucumber Kheera Hybrid Seeds - Kakdi -,20.0,Hybrid,Rainy Season,44.0,4.27,69.0,345.00,99.0,495.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,495.0
3,Coriander Dhaniya Seeds -,1000.0,Open Pollinated,All Season,88.0,4.77,59.0,5.90,99.0,9.9,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,9.9
4,Spinach Palak Seeds,750.0,Open Pollinated,All Season,81.0,4.72,65.0,8.67,99.0,13.2,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,13.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green Thick wall fruits...,50.0,Open Pollinated,All Season,2.0,1.00,79.0,158.00,149.0,298.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,298.0
1128,Bitter Gourd Barahmasi Seeds - karela,50.0,Open Pollinated,All Season,5.0,4.20,59.0,118.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.0
1129,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi,50.0,Open Pollinated,All Season,2.0,4.00,69.0,138.00,149.0,298.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,298.0
1130,Collard Seeds,50.0,Open Pollinated,All Season,3.0,5.00,49.0,98.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.0


In [152]:
feature_selected_df.columns

Index(['Seed Name', 'Number of Seeds', 'Seed Type', 'Sowing Season',
       'No_of_Reviews', 'Average_Rating', 'Sale_Price',
       'Sale_Price_per_100_Seeds', 'Regular_Price', 'Discount',
       'Discount_Percent', 'Is_All', 'Is_Best_Seller', 'Is_Bulbs',
       'Is_Exotic_Seeds', 'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds',
       'Is_Organic_Seeds', 'Is_Other_Seeds', 'Is_Rainy_Seeds',
       'Is_Rainy_Flower_Seeds', 'Is_Summer_Flower_Seeds',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter_Flower_Seeds',
       'Is_Winter_Seeds', 'Is_Summer', 'Is_Winter', 'Is_Rainy'],
      dtype='object')

In [166]:
feature_selected_df['Regular_Price_per_100_Seeds'] = (
    
    (
        feature_selected_df['Regular_Price']
        
        / feature_selected_df['Number of Seeds'].replace(0, np.nan)
    )
    
    * 100
    
).round(2)

In [154]:
feature_selected_df.columns

Index(['Seed Name', 'Number of Seeds', 'Seed Type', 'Sowing Season',
       'No_of_Reviews', 'Average_Rating', 'Sale_Price',
       'Sale_Price_per_100_Seeds', 'Regular_Price', 'Discount',
       'Discount_Percent', 'Is_All', 'Is_Best_Seller', 'Is_Bulbs',
       'Is_Exotic_Seeds', 'Is_Fruit_Seeds', 'Is_Herb_Seeds', 'Is_Micro_Seeds',
       'Is_Organic_Seeds', 'Is_Other_Seeds', 'Is_Rainy_Seeds',
       'Is_Rainy_Flower_Seeds', 'Is_Summer_Flower_Seeds',
       'Is_Summer_Vegetable_Seeds', 'Is_Winter_Flower_Seeds',
       'Is_Winter_Seeds', 'Is_Summer', 'Is_Winter', 'Is_Rainy',
       'Regular_Price_per_100_Seeds'],
      dtype='object')

In [167]:
feature_selected_df

,Seed Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Regular_Price_per_100_Seeds
0,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,138.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.0
1,Okra or Lady Finger Hybrid bhindi Seeds -,50.0,Hybrid,Rainy Season,73.0,4.30,65.0,130.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,198.0
2,Cucumber Kheera Hybrid Seeds - Kakdi -,20.0,Hybrid,Rainy Season,44.0,4.27,69.0,345.00,99.0,495.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,495.0
3,Coriander Dhaniya Seeds -,1000.0,Open Pollinated,All Season,88.0,4.77,59.0,5.90,99.0,9.9,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,9.9
4,Spinach Palak Seeds,750.0,Open Pollinated,All Season,81.0,4.72,65.0,8.67,99.0,13.2,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,13.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,Chilli Yellowish Light Green Thick wall fruits...,50.0,Open Pollinated,All Season,2.0,1.00,79.0,158.00,149.0,298.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,298.0
1128,Bitter Gourd Barahmasi Seeds - karela,50.0,Open Pollinated,All Season,5.0,4.20,59.0,118.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.0
1129,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi,50.0,Open Pollinated,All Season,2.0,4.00,69.0,138.00,149.0,298.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,298.0
1130,Collard Seeds,50.0,Open Pollinated,All Season,3.0,5.00,49.0,98.00,99.0,198.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.0


In [175]:
feature_selected_df[feature_selected_df['Is_Summer'] == 1]

,Seed Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Regular_Price_per_100_Seeds
0,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,138.00,99.0,198.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.00
3,Coriander Dhaniya Seeds -,1000.0,Open Pollinated,All Season,88.0,4.77,59.0,5.90,99.0,9.90,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,9.90
4,Spinach Palak Seeds,750.0,Open Pollinated,All Season,81.0,4.72,65.0,8.67,99.0,13.20,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,13.20
5,Brinjal Seeds Black -,50.0,Open Pollinated,All Season,48.0,4.15,69.0,138.00,99.0,198.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.00
6,Capsicum Hari Shimla Mirch Seeds F1 Hybrid -,30.0,F-1 Hybrid,All Season,58.0,4.69,120.0,400.00,149.0,496.67,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,496.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1124,Spinach Malabar Poi Saag Seeds,50.0,Open Pollinated,All Season,2.0,5.00,69.0,138.00,199.0,398.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,398.00
1127,Chilli Yellowish Light Green Thick wall fruits...,50.0,Open Pollinated,All Season,2.0,1.00,79.0,158.00,149.0,298.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,298.00
1128,Bitter Gourd Barahmasi Seeds - karela,50.0,Open Pollinated,All Season,5.0,4.20,59.0,118.00,99.0,198.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.00
1129,Salad Cabbage F1 Hybrid Seeds - Patta Gobhi,50.0,Open Pollinated,All Season,2.0,4.00,69.0,138.00,149.0,298.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,298.00


In [172]:
cleaned_summer_flower_seeds

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Number of Seeds,Seed Type,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Product_URL,Is_Summer_Flower_Seeds
0,Marigold Orange Flower Seeds - 100 Seeds (Gend...,43.0,4.67,65.0,99.0,34.0,34.34,100 Seeds,"Open Pollinated,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"24x9 inch, 18x6 inch, 24x6 inch, 24x12 inch",Easy – Ideal for beginners,https://organicbazar.net/products/marigold-see...,1
1,Zinnia Mix Color Flower Seeds (100 Seeds) High...,59.0,4.63,76.0,110.0,34.0,30.91,100 Seeds,"Open Pollinated,",All Season,18-25°C,6-10 Days,Above 70%,45-60 Days,"24x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/zinnia-doubl...,1
2,Portulaca Grandiflora (Moss Rose) Double Mix S...,44.0,4.18,85.0,120.0,35.0,29.17,500 Seeds,"Open Pollinated,","Rainy Season,",18-30 °C,10- 15 Days,Above 70%,45-60 Days,"18x6 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/portulaca-gr...,1
3,Petunia Dwarf Stars Mix Seeds (300 Seeds) High...,32.0,4.19,49.0,90.0,41.0,45.56,300 Seeds,"Open Pollinated,","Winter Season,",18-25°C,7–14 Days,Above 70%,60-80 Days,"18x9 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/petunia-dwar...,1
4,Marigold Yellow Flower Seeds - 100 Seeds (Pila...,26.0,4.12,65.0,99.0,34.0,34.34,100 Seeds,"Open Pollinated,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/marigold-see...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Cosmos Sensation Dwarf Red Flower Seeds (100 S...,12.5,0.00,59.0,110.0,51.0,46.36,100 Seeds,"Open Pollinated,",All Season,18-25°C,7–14 Days,Above 70%,45-60 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/cosmos-sensa...,1
66,French Marigold Durango Bolero Flower Seeds (3...,12.5,0.00,65.0,149.0,84.0,56.38,30 Seeds,"Hybrid,",All Season,15-26 °C,6-14 Days,Above 70%,45-60 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/french-marig...,1
67,Celosia Ice Cream Mix Color Flower Seeds (50 S...,12.5,0.00,143.0,269.0,126.0,46.84,50 Seeds,"Hybrid,","Summer Season,",18-30 °C,7–14 Days,Above 70%,60-80 Days,"15x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/celosia-ice-...,1
68,Salvia Vista Red Flower Seeds (20 Seeds) High ...,12.5,0.00,65.0,149.0,84.0,56.38,20 Seeds,"Open Pollinated,","Winter Season,",18–28°C,7-21 Days,Above 70%,60-80 Days,"18x12 inch,",Easy – Ideal for beginners,https://organicbazar.net/products/salvia-vista...,1


In [184]:
feature_selected_df[feature_selected_df['Average_Rating'] == 3.5]

,Seed Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Regular_Price_per_100_Seeds
184,Pumpkin Oval seeds - Kaddu,12.0,Hybrid,All Season,4.0,3.5,59.0,491.67,110.0,916.67,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,916.67
346,Hyacinth Mixed Color Flower Bulbs 05 Bulbs,5.0,Open Pollinated,Winter Season,2.0,3.5,599.0,11980.00,999.0,19980.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,19980.00
490,Spinach Microgreen Seeds 25g,25.0,Untreated,All Season,2.0,3.5,49.0,196.00,90.0,360.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,360.00
1001,Pumpkin Oval seeds - Kaddu,12.0,Hybrid,All Season,4.0,3.5,59.0,491.67,110.0,916.67,...,0.0,0.0,0.0,0.0,1.0,0.0,1,1,1,916.67


In [194]:
cleaned_bulbs[cleaned_bulbs['Name_of_Seed']=='Hyacinth Mixed Color Flower Bulbs 05 Bulbs']

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Product_URL,Number of Seeds,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Seed Type,Is_Bulbs


In [198]:
cleaned_bulbs[cleaned_bulbs['Average_Rating'] == 3.5]

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Product_URL,Number of Seeds,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Seed Type,Is_Bulbs
15,Hyacinth Mixed Color Flower Bulbs (05 Bulbs),2.0,3.5,599.0,999.0,400.0,40.04,https://organicbazar.net/products/hyacinth-mix...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"15x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1


In [201]:
feature_selected_df[feature_selected_df['Is_Bulbs'] == 0]

,Seed Name,Number of Seeds,Seed Type,Sowing Season,No_of_Reviews,Average_Rating,Sale_Price,Sale_Price_per_100_Seeds,Regular_Price,Discount,...,Is_Rainy_Seeds,Is_Rainy_Flower_Seeds,Is_Summer_Flower_Seeds,Is_Summer_Vegetable_Seeds,Is_Winter_Flower_Seeds,Is_Winter_Seeds,Is_Summer,Is_Winter,Is_Rainy,Regular_Price_per_100_Seeds
275,Tomato F1 Hybrid Seeds - Tamatar,50.0,F-1 Hybrid,All Season,103.0,4.11,69.0,138.00,99.0,198.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.00
276,Okra or Lady Finger Hybrid bhindi Seeds -,50.0,Hybrid,Rainy Season,73.0,4.30,65.0,130.00,99.0,198.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,198.00
277,OrganicBazar 1212 HDPE Round Grow Bag Premium ...,50.0,Open Pollinated,All Season,302.0,4.63,799.0,1598.00,1990.0,3980.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,3980.00
278,Cucumber Kheera Hybrid Seeds - Kakdi -,20.0,Hybrid,Rainy Season,44.0,4.27,69.0,345.00,99.0,495.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,495.00
279,Coriander Dhaniya Seeds -,1000.0,Open Pollinated,All Season,88.0,4.77,59.0,5.90,99.0,9.90,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,9.90
280,Spinach Palak Seeds,750.0,Open Pollinated,All Season,81.0,4.72,65.0,8.67,99.0,13.20,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,13.20
281,Brinjal Seeds Black -,50.0,Open Pollinated,All Season,48.0,4.15,69.0,138.00,99.0,198.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,198.00
282,Capsicum Hari Shimla Mirch Seeds F1 Hybrid -,30.0,F-1 Hybrid,All Season,58.0,4.69,120.0,400.00,149.0,496.67,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,496.67
283,OrganicBazar 15x15 Grow Bag for Vegetable Gard...,50.0,Open Pollinated,All Season,140.0,4.61,699.0,1398.00,999.0,1998.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,1998.00
284,French Beans Seeds -,20.0,Open Pollinated,All Season,37.0,4.32,69.0,345.00,99.0,495.00,...,0.0,0.0,0.0,0.0,0.0,0.0,1,1,1,495.00


In [202]:
cleaned_bulbs

,Name_of_Seed,No_of_Reviews,Average_Rating,Sale_Price,Regular_Price,Discount,Discount_Percent,Product_URL,Number of Seeds,Sowing Season,Germination Temperature,Germination Time,Germination Rate,Blooming Time,Container Requirement / Pots,Growing Level,Seed Type,Is_Bulbs
0,Original Black Turmeric (Kali Haldi ke beej) F...,20.0,4.65,299.0,799.0,500.0,62.58,https://organicbazar.net/products/black-turmer...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
1,Ranunculus Flower Bulbs Mixed Colors (5N),23.0,3.87,399.0,599.0,200.0,33.39,https://organicbazar.net/products/ranunculus-f...,05 Bulbs,"Winter Season,",10-15°C.,16-30 Days,Above 70%,90-120 Days,"15x15 inch,",Hard - Suitable for experienced gardeners,"Open Pollinated,",1
2,Rain Lily Flower Bulbs Mixed Color (10 Bulbs) ...,22.0,4.32,169.0,199.0,30.0,15.08,https://organicbazar.net/products/rain-lily-fl...,10 Bulbs,"Rainy Season,",22-30 °C.,14-28 Days,Above 70%,60-90 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
3,Kashmiri Saffron (Kesar) Bulbs 10N,23.0,4.48,499.0,599.0,100.0,16.69,https://organicbazar.net/products/kashmiri-saf...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
4,Gladiolus Mix Color Flower Bulbs (10N),21.0,4.71,199.0,299.0,100.0,33.44,https://organicbazar.net/products/gladiolus-fl...,10 Bulbs,"Winter Season,",12–18°C,14-28 Days,Above 70%,60-90 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,Tulip Doberman (Black Purple) Flower Bulbs,2.0,0.00,450.0,999.0,549.0,54.95,https://organicbazar.net/products/tulip-doberm...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
88,Safed Musli Bulbs Seeds,1.0,5.00,99.0,149.0,50.0,33.56,https://organicbazar.net/products/safed-musli-...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
89,Gladiolus Yellow Flower Bulbs,2.0,0.00,99.0,199.0,100.0,50.25,https://organicbazar.net/products/gladiolus-ye...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1
90,Asiatic (Lilium) Lily Yellow Flower Bulbs,2.0,0.00,299.0,399.0,100.0,25.06,https://organicbazar.net/products/asiatic-lili...,05 Bulbs,"Winter Season,",10-15°C.,14-28 Days,Above 70%,90-120 Days,"12x12 inch,",Easy – Ideal for beginners,"Open Pollinated,",1


In [205]:
feature_selected_df.loc[
    
    feature_selected_df['Seed Name']
    
    .str.contains('Bulbs', case=False, na=False),
    
    'Is_Bulbs'
    
] = 1
feature_selected_df['Is_Bulbs'] = (
    
    feature_selected_df['Is_Bulbs']
    
    .fillna(0)
    
    .astype(int)
)

In [209]:
feature_selected_df['Discount_Percent']

0       30.0
1       34.0
2       30.0
3       40.0
4       34.0
        ... 
1127    70.0
1128    40.0
1129    80.0
1130    50.0
1131    40.0
Name: Discount_Percent, Length: 1132, dtype: float64

## Using Selenium

In [210]:
from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.service import Service

import pandas as pd

import time

In [212]:

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import re
import selenium
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys


In [219]:
option = Options()
option.add_argument('--start-maximized') 
option.add_argument('--disable-blink-features=AutomationControlled')
option.add_argument('--disable-notifications') 
option.add_argument("--disable-popup-blocking")


In [220]:
driver = webdriver.Chrome(options = option)

driver.maximize_window()

driver.get(
    "https://organicbazar.net/"
)

In [221]:
driver.get(
    "https://organicbazar.net/collections/organic-seeds"
)

In [222]:

all_products = []

# Count cards once
total_cards = len(

    driver.find_elements(
        By.CSS_SELECTOR,
        "h3.card__heading a"
    )
)

print("Total Cards:", total_cards)

for i in range(total_cards):

    # Re-find cards every iteration
    cards = driver.find_elements(
        By.CSS_SELECTOR,
        "h3.card__heading a"
    )

    cards[i].click()

    time.sleep(2)

    product_data = {}

    # =====================
    # TITLE
    # =====================

    try:

        product_data["Name_of_Seed"] = driver.find_element(

            By.TAG_NAME,

            "h1"

        ).text

    except:

        product_data["Name_of_Seed"] = None

    # =====================
    # SALE PRICE
    # =====================

    try:

        product_data["Sale_Price"] = driver.find_element(

            By.CSS_SELECTOR,

            "span.price-item--sale"

        ).text

    except:

        product_data["Sale_Price"] = None

    # =====================
    # REGULAR PRICE
    # =====================

    try:

        product_data["Regular_Price"] = driver.find_element(

            By.CSS_SELECTOR,

            "s.price-item--regular"

        ).text

    except:

        product_data["Regular_Price"] = None

    # =====================
    # SPECIFICATION TABLE
    # =====================

    rows = driver.find_elements(

        By.CSS_SELECTOR,

        "table tr"
    )

    for row in rows:

        cols = row.find_elements(
            By.TAG_NAME,
            "td"
        )

        if len(cols) == 2:

            key = cols[0].text.strip()

            value = cols[1].text.strip()

            product_data[key] = value

    all_products.append(
        product_data
    )

    print(
        f"Completed {i+1}/{total_cards}"
    )

    # =====================
    # GO BACK
    # =====================

    driver.back()

    time.sleep(2)

df = pd.DataFrame(
    all_products
)

df.to_csv(
    "organic_seeds.csv",
    index=False
)

driver.quit()

Total Cards: 28


ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6818f7de5+14895]
	chromedriver!GetHandleVerifier [0x7ff6818f7e50+14900]
	chromedriver!(No symbol) [0x7ff68165d396]
	chromedriver!(No symbol) [0x7ff6816b8dc4]
	chromedriver!(No symbol) [0x7ff6816abce6]
	chromedriver!(No symbol) [0x7ff6816e005a]
	chromedriver!(No symbol) [0x7ff6816ab566]
	chromedriver!(No symbol) [0x7ff68170486f]
	chromedriver!(No symbol) [0x7ff6816a9df8]
	chromedriver!(No symbol) [0x7ff6816aace3]
	chromedriver!GetHandleVerifier [0x7ff681c0cc49+3296f9]
	chromedriver!GetHandleVerifier [0x7ff681c07375+323e25]
	chromedriver!GetHandleVerifier [0x7ff681c2bc82+348732]
	chromedriver!GetHandleVerifier [0x7ff681916045+32af5]
	chromedriver!GetHandleVerifier [0x7ff68191ecec+3b79